In [1]:

!pip install numpy pandas matplotlib
!pip install torch
!pip install torchvision
!pip install torchmetrics
!pip install LTNtorch

testes

In [3]:


import ltn
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

from PIL import Image
import os

import torch
from torch import optim
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torch.utils.data import Subset
#from tqdm import tqdm

# !pip install torchvision
import torchvision

import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.models as models
from datetime import datetime
import importlib
import subprocess
import json
#from utils.axioms import compute_axioms, get_number_of_final_classes, get_CLASSES
#from utils.occlusion import occlusion_map, show_occlusion


# !pip install torchmetrics
import torchmetrics

""" HYPERPARAMETERS """


DATA_ROOT = Path("/home/sofia/Desktop/datasets/datasetVCB")
#DATA_ROOT = Path("/home/sofia/Desktop/datasets/newDataset1/full_run/")
BATCH_SIZE = 32
COMPLEXITY_DATASET = 1
EPOCHS = 10
TOTAL_EXAMPLES = 20000
#TRAIN_TEST_LIMIT = 16000
LEARNING_RATE = 0.0001
csv_path = "results.csv"
txt_path = "stats.txt"
confusion_matrix_path = "conf_matrix.txt"

In [6]:
BATCH_SIZE = 32
LIMIT = 1500
EPOCHS = 150
EARLY_STOPPING_DELTA = 0.005
PATIENCE = 10
MODEL_SAVING_INTERVAL = 100
DEBUG_MODE = False

In [5]:
import torch

print("torch works")
x = torch.randn(1000,1000)
print(x.mean())

torch works
tensor(0.0001)


CNN with multiple Loss

In [14]:
TRAIN_WITH_A_SUBSET = True
SUBSET_SIZE = 12000
STRATIFY = True
SUBSET_SEED = 42
VALIDATION_INTERVAL = 2
#default 42

In [ ]:
print(f"Starting execution using DEBUG_MODE: {DEBUG_MODE}")
print(f"Starting execution using TRAIN_WITH_A_SUBSET: {TRAIN_WITH_A_SUBSET}")

if TRAIN_WITH_A_SUBSET: print(f"SUBSET SIZE: {SUBSET_SIZE}, SUBSET_SEED: {SUBSET_SEED}, STRATIFY:{STRATIFY} ")

#print("pode ser necessario de reiniciar duas vezes apos mudar o conhecimento no dicionario das regras\n")

#from utils.axioms import compute_axioms, get_number_of_final_classes, get_CLASSES
import utils.axioms as axioms
import utils.occlusion as occlusion
import utils.rules_and_concepts as rules_and_concepts
import utils.test_new_rules as test_new_rules
#import charter
importlib.reload(rules_and_concepts)
importlib.reload(axioms)
importlib.reload(occlusion)
importlib.reload(test_new_rules)
#importlib.reload(charter)



data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(DATA_ROOT, "all.csv")
images = Path(DATA_ROOT, "c1")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]

CLASSES = rules_and_concepts.get_all_classes() #era get_classes

ALL_EXISTING_CLASSES = rules_and_concepts.get_all_classes()
ALL_CONCEPTS = rules_and_concepts.get_ALL_CONCEPTS()
final_classes = len(rules_and_concepts.get_all_final_classes())

assert CLASSES[:final_classes] == rules_and_concepts.get_all_final_classes(), \
    f"Final classes mismatch: {CLASSES[:final_classes]}"
assert len(CLASSES) <= final_classes + len(ALL_CONCEPTS) + 5, \
    f"CLASSES too long for model output: {len(CLASSES)} vs {final_classes + len(ALL_CONCEPTS) + 5}"
print(f"[DEBUG] CLASSES order: {CLASSES}")
print(f"[DEBUG] Final classes (first {final_classes}): {CLASSES[:final_classes]}")
print(f"[DEBUG] Concept classes (after): {CLASSES[final_classes:]}")

#print(f"final classes: {final_classes}, number of classes {len(CLASSES)}, CLASSES in main file: {CLASSES}")

class BuildingsDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = self.img_dir / row["id"]
        image = Image.open(img_path).convert("RGB")



        #target = torch.tensor(row[CLASSES].values, dtype=torch.float32)
        target = torch.tensor(row[CLASSES].to_numpy(dtype=float), dtype=torch.float32)


        if self.transform:
            image = self.transform(image)

        return image, target, row["id"]


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

def random_subset(dataset, n, seed=42):
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(dataset), size=n, replace=False)
    return Subset(dataset, indices.tolist())


def stratified_subset(dataset, df, n, stratify_class, seed=42):
    """Stratify on a single (usually rare) class to preserve its positive rate."""
    rng = np.random.default_rng(seed)
    labels = df[stratify_class].to_numpy()  # fast: no image loading
    pos = np.where(labels == 1)[0]
    neg = np.where(labels == 0)[0]

    pos_ratio = len(pos) / len(labels)
    n_pos = int(round(n * pos_ratio))
    n_neg = n - n_pos

    # Guard against asking for more than exists
    n_pos = min(n_pos, len(pos))
    n_neg = min(n_neg, len(neg))

    chosen = np.concatenate([
        rng.choice(pos, size=n_pos, replace=False),
        rng.choice(neg, size=n_neg, replace=False),
    ])
    rng.shuffle(chosen)  # mix pos/neg so batches aren't blocky
    return Subset(dataset, chosen.tolist())


TRAIN_TEST_LIMIT = int(0.8 * len(df))

train_df = df[:TRAIN_TEST_LIMIT]
test_df = df[TRAIN_TEST_LIMIT:]

df[CLASSES] = df[CLASSES].apply(pd.to_numeric)


train_dataset = BuildingsDataset(train_df, images, transform=transform)
LEN_TRAIN_DATASET= len(train_dataset)
test_dataset = BuildingsDataset(test_df, images, transform=transform)

subset_class_pos_rates = ""
info_about_dataset = ""
train_set_df =train_df

if TRAIN_WITH_A_SUBSET:
    stratified_class = "Machine"

    if STRATIFY:

        train_subset = stratified_subset(
            train_dataset, train_df,
            n=SUBSET_SIZE,
            stratify_class=stratified_class,   # or your rarest class
            seed=SUBSET_SEED,
        )
        #train_set = train_subset
    else:
        train_subset = random_subset(train_dataset, n=SUBSET_SIZE, seed=SUBSET_SEED)
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)

    print(f"[subset] using {len(train_subset)}/{len(train_dataset)} train samples (stratified={STRATIFY}, stratified_class={stratified_class}, seed={SUBSET_SEED})")
    info_about_dataset = f"[subset] using {len(train_subset)}/{len(train_dataset)} train samples (stratified={STRATIFY}, seed={SUBSET_SEED})"
    
    subset_df = train_df.iloc[train_subset.indices]
    train_set_df = subset_df
    print("[subset] class positive rates:")
    print(subset_df[CLASSES].mean().to_string())
    subset_class_pos_rates = subset_df[CLASSES].mean().to_string()
else:
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)





#print(train_df["Cafe"])

"""CNN MODEL"""

class CNN(nn.Module):
    def __init__(self, in_channels, num_classes, name="cnn_model"):

        """
        Building blocks of convolutional neural network.

        Parameters:
            * in_channels: Number of channels in the input image (for grayscale images, 1)
            * num_classes: Number of classes to predict. In our problem, 10 (i.e digits from  0 to 9).
        """
        super(CNN, self).__init__()

        #nbr_of_concepts = len(CLASSES) - final_classes

        nbr_of_neurons = min(len(ALL_CONCEPTS) + 5, 48)

        #layers #era 16,32,64
        self.conv1a = nn.Conv2d(in_channels=3, out_channels=8, kernel_size=3)
        self.conv1b = nn.Conv2d(in_channels=8, out_channels=12, kernel_size=3)

        self.conv2a = nn.Conv2d(in_channels=12, out_channels=16, kernel_size=3)
        self.conv2b = nn.Conv2d(in_channels=16, out_channels=24, kernel_size=3)

        self.conv3a = nn.Conv2d(in_channels=24, out_channels=32, kernel_size=3)
        #self.conv3b = nn.Conv2d(in_channels=24, out_channels=32, kernel_size=3)
        #self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3)

        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(0.5)

        #3 conv 43624, 4 conv 18432 com 6 é 13824
        size = self._get_flattened_size() 
        print(f"size of CNN {size}, concept layer: {nbr_of_neurons}")
        self.dense1 = nn.Linear(size, 128)
        self.dense2 = nn.Linear(128, 64)

        self.concept_layer = nn.Linear(64, nbr_of_neurons)

        self.output_layer = nn.Linear(nbr_of_neurons, num_classes)


    def _get_flattened_size(self):
        with torch.no_grad():
            x = torch.zeros(1, 3, 224, 224)
            x = F.relu(self.conv1a(x))
            x = F.relu(self.conv1b(x))
            x = self.maxpool(x)

            x = F.relu(self.conv2a(x))
            x = F.relu(self.conv2b(x))
            x = self.maxpool(x)

            x = F.relu(self.conv3a(x))
            #x = F.relu(self.conv3b(x))
            x = self.maxpool(x)

            return x.view(1, -1).shape[1]


    def forward(self, inputs):
        """
        Define the forward pass of the neural network.

        Parameters:
            x: Input tensor.

        Returns:
            torch.Tensor
                The output tensor after passing through the network.
        """
        if isinstance(inputs, (list, tuple)):
            inputs = inputs[0]


        x = F.relu(self.conv1a(inputs))
        x = F.relu(self.conv1b(x))
        x = self.maxpool(x)

        x = F.relu(self.conv2a(x))
        #x = self.maxpool(x)

        x = F.relu(self.conv2b(x))
        x = self.maxpool(x)
        
        x = F.relu(self.conv3a(x))
        #x = F.relu(self.conv3b(x))
        x = self.maxpool(x)

        x = self.flatten(x)
        x = self.dropout(x)

        

        x = F.leaky_relu(self.dense1(x))
        x = self.dropout(x)
        x = F.leaky_relu(self.dense2(x))

        y = F.leaky_relu(self.concept_layer(x))

        x = self.output_layer(y)

        result = torch.cat([x, y], dim=1)
        return result

      
'''
class CNN(nn.Module):
    def __init__(self, in_channels, num_classes, name="cnn_model"):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, 3)
        self.conv2 = nn.Conv2d(16, 32, 3)
        self.conv3 = nn.Conv2d(32, 64, 3)

        self.pool = nn.MaxPool2d(2)

        self.gap = nn.AdaptiveAvgPool2d((1,1))

        self.fc1 = nn.Linear(64, 128)
        self.fc2 = nn.Linear(128, 64)

        self.concept_layer = nn.Linear(64, 32)
        self.output_layer = nn.Linear(32, num_classes)

    def forward(self, x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = self.gap(x)
        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        y = F.relu(self.concept_layer(x))
        x = self.output_layer(y)

        return torch.cat([x, y], dim=1)

'''
class PredicateModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.activation = nn.Sigmoid()

    def forward(self, logits, class_idx):
        probs = self.activation(logits)  # [batch, num_classes]

        # Make sure class_idx is the same shape as batch
        if class_idx.dim() == 0:
            class_idx = class_idx.repeat(logits.shape[0])

        # Use torch.arange to select batch elements along dim 0
        selected = probs[torch.arange(probs.size(0)), class_idx.long()]

        return selected  # [batch]



# Predicates
#final classes = 4
def reset_model(in_channels, num_classes, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)

    #print("\n\n\n\n problem in reset model??? \n\n\n\n")

    logits_model = CNN(in_channels=in_channels, num_classes=num_classes)#3, final classes
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logits_model = logits_model.to(device)

    # compute pos_weight from training data
    class_cols = CLASSES[:final_classes]
    print(f"class_cols: {class_cols}")
    pos_counts = train_set_df[class_cols].sum()
    neg_counts = len(train_set_df) - pos_counts
    weights = (neg_counts / pos_counts.clip(lower=1)).values
    pos_weight = torch.tensor(weights, dtype=torch.float32).to(device)


    #missCounter = 0
    #pos_weight é basicamente a razão entre o numero de casos positvos e os negativos. a ideia é que o modelo deve penalizar um falso errado proporicionalmente mais,
    #porque existe mais errados do que falsos
    #So for Restaurant (12% positive, 88% negative), the weight would be ~7.2. Missing a "this is a Restaurant" prediction now costs 7.2x more than incorrectly predicting "this is a Restaurant" when it isn't.
    classification_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    p = ltn.Predicate(PredicateModel())

    return logits_model, device,classification_loss_fn, p


def __get_elements_from_batch(batch_elements):
    #print("\n\n\n problems in get_elems? \n\n\n")
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    #labels_list = np.transpose(inverted_labels_list)
    labels_list = inverted_labels_list.T
    return features_calculated, labels_list



""" TRAINING FUNCTIONS """

class Counter:
    def __init__(self):
        self.reset_state()

    def reset_state(self):
        self.value = 0.0

    def update(self, value):
        self.value = value

    def result(self):
        return self.value

class MeanMetric:
    def __init__(self):
        self.reset_state()

    def update(self, value, n=1):
        self.total += value * n
        self.count += n

    def compute(self):
        return self.total / self.count if self.count != 0 else 0.0


    def reset_state(self):
        self.total = 0.0
        self.count = 0

    def result(self):
        return self.compute()
    

class RunningAverage:
    def __init__(self):
        self.reset_state()

    def reset_state(self):
        self.total = 0
        self.count = 0

    def update(self, value):
        self.total += value
        self.count += 1

    
    def result(self):
        if self.count == 0:
            return 0
        return self.total / self.count


metrics_dict = {
    "train_sat_kb": MeanMetric(),
    "val_sat_kb": MeanMetric(),
    "train_loss": RunningAverage(),
    "val_loss": RunningAverage(),
    "combined_val_loss": RunningAverage(),
    "combined_train_loss": RunningAverage(),
    "alpha": Counter()
}

class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": MeanMetric()
    for class_name in CLASSES
}

metrics_dict.update(class_train_accuracy_metric)

class_val_accuracy_metric = {
    f"{class_name.lower()}_val_accuracy": MeanMetric()
    for class_name in CLASSES
}
metrics_dict.update(class_val_accuracy_metric)


'''
classes_for_confusion_matrix = [
    "Awning",
    "Table",
    "VendingMachine",
    "Statue",
]
'''

classes_for_confusion_matrix = rules_and_concepts.get_classes()


conf_matrices = {
    f"{cls}_train": torch.zeros(2, 2, dtype=torch.int32)
    for cls in classes_for_confusion_matrix
}

conf_matrices.update({
    f"{cls}_val": torch.zeros(2, 2, dtype=torch.int32)
    for cls in classes_for_confusion_matrix
})




#optimizer = torch.optim.Adam(logits_model.parameters(), lr=LEARNING_RATE)


def train_step(features_param, *labels_list, optimizer, logits_model, classification_loss_fn, p, alpha=0.5):

    if not hasattr(train_step, "_verified"):
        train_step._verified = True
        print(f"[DEBUG] labels_list length: {len(labels_list)}, final_classes: {final_classes}")
        print(f"[DEBUG] class labels map to: {CLASSES[:final_classes]}")
        print(f"[DEBUG] concept labels map to: {CLASSES[final_classes:]}")
        logits = logits_model(features_param)
        print(f"[DEBUG] logits shape: {logits.shape}, expected: [batch, {final_classes + min(len(ALL_CONCEPTS)+5, 48)}]")


    logits_model.train()
    optimizer.zero_grad()

    logits = logits_model(features_param)

    # first 4 logits correspond to classes
    class_logits = logits[:, :final_classes]

    # only first 4 labels
    class_labels = labels_list[:final_classes]
    labels_tensor = torch.stack(class_labels, dim=1).float()
    complete_lables_tensor = torch.stack(labels_list, dim=1).float()

    #print(f"\n\n\n class logits size {class_logits.size()}, labels tensor size: {labels_tensor.size()} class_labesl size: {len(class_labels)} \n\n\n")

    classification_loss = classification_loss_fn(class_logits, labels_tensor)

    sat = axioms.compute_axioms(logits, *labels_list, p=p, debug_mode=DEBUG_MODE, validation_mode=False)

    ltn_loss = 1.0 - sat

    # ----- Combined loss -----



    loss = alpha * classification_loss + (1 - alpha) * ltn_loss

    loss.backward()
    optimizer.step()

    print(f"Sat Function Result: {sat.item()}, Class_loss: {classification_loss}, ltn-loss: {ltn_loss}, Combined_loss:{loss} ")

    # Update metric
    metrics_dict["train_sat_kb"].update(sat.item())
    metrics_dict["train_loss"].update(classification_loss.item())
    metrics_dict["combined_train_loss"].update(loss.item())

    #predictions = logits

    calculate_accuracy_for_each_class("train", logits, labels_list)
    update_confusion_matrix("train", logits.detach(), complete_lables_tensor, CLASSES, conf_matrices)

@torch.no_grad()
def test_step(features_param, *labels_list, logits_model, classification_loss_fn, p, alpha=0.5):
    logits_model.eval()

    logits = logits_model(features_param)
        
    # first 4 logits correspond to classes
    class_logits = logits[:, :final_classes]

    # only first 4 labels
    class_labels = labels_list[:final_classes]
    labels_tensor = torch.stack(class_labels, dim=1).float()
    complete_lables_tensor = torch.stack(labels_list, dim=1).float()

    classification_loss = classification_loss_fn(class_logits, labels_tensor)

    sat_kb = axioms.compute_axioms(logits, *labels_list, p=p, debug_mode=DEBUG_MODE, validation_mode=True)

    ltn_loss = 1.0 - sat_kb

    # ----- Combined loss -----
    loss = alpha * classification_loss + (1 - alpha) * ltn_loss


    metrics_dict["val_sat_kb"].update(sat_kb.item())
    metrics_dict["val_loss"].update(classification_loss.item())
    metrics_dict["combined_val_loss"].update(loss.item())

    calculate_accuracy_for_each_class("val", logits, labels_list)
    update_confusion_matrix("val", logits.detach(), complete_lables_tensor, CLASSES, conf_matrices)


""" AUXILIARY FUNCTIONS """
def update_confusion_matrix(phase, logits, labels, class_names, conf_matrices):
    """
    predictions: tensor (batch_size, num_classes) -> logits
    labels: tensor (batch_size, num_classes)
    """
    probs = torch.sigmoid(logits[:, :len(class_names)])
    preds = (probs > 0.5).int()

    for i, class_name in enumerate(class_names):
        if class_name not in classes_for_confusion_matrix:
            continue

        y_true = labels[:, i].int()
        y_pred = preds[:, i]

        # Compute confusion matrix manually
        tn = torch.sum((y_true == 0) & (y_pred == 0)) #TN
        fp = torch.sum((y_true == 0) & (y_pred == 1)) #FP
        fn = torch.sum((y_true == 1) & (y_pred == 0)) #FN
        tp = torch.sum((y_true == 1) & (y_pred == 1)) 

        cm = torch.tensor([[tn, fp],
                           [fn, tp]], dtype=torch.int32)

        conf_matrices[f"{class_name}_{phase}"] += cm


def reset_conf_matrices(conf_matrices):
    for key in conf_matrices:
        conf_matrices[key].zero_()

def print_conf_matrices(conf_matrices):
    for key, cm in conf_matrices.items():
        print(f"{key} confusion matrix:\n{cm}\n")


def calculate_accuracy_for_each_class(phase, logits, labels_param):
    probs = torch.sigmoid(logits)

    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i].float()

        y_pred = probs[:, i]
        y_pred = (y_pred > 0.5).float()

        accuracy = (y_pred == class_labels).float().mean()

        metrics_dict[f"{class_name.lower()}_{phase}_accuracy"].update(
            accuracy.item()
        )


'''CLINGO'''

def export_predictions_to_csv_and_lp(model_path, output_csv_path, output_lp_path, sample_loader=test_loader):

    logits_model, device, _, p = reset_model(in_channels=3, num_classes=final_classes)

    logits_model.load_state_dict(torch.load(model_path))
    logits_model.to(device)
    logits_model.eval() #CNN model in eval mode. if its in train mode, the dropout layers mess up the results bc they have different values

    rows = []

    with torch.no_grad():
        for batch_elements in sample_loader:
            features_param, _ = __get_elements_from_batch(batch_elements)
            ids = batch_elements[2]

            features_param = features_param.to(device)

            logits = logits_model(features_param)
            probs = torch.sigmoid(logits)

            # move to cpu
            #probs = probs.cpu().numpy()

            for j, sample in enumerate(probs):
                sid = axioms.make_sid(ids[j])
                row = {"sample_id": sid}
                for i, class_name in enumerate(CLASSES):
                    row[class_name] = int(sample[i] > 0.5)
                rows.append(row)

    df_out = pd.DataFrame(rows)
    df_out.to_csv(output_csv_path, index=False)

    #df_out = pd.read_csv(csv_path)

    with open(output_lp_path, "w") as f:
        for _, row in df_out.iterrows():
            sid = row["sample_id"]
            f.write(f"sample({sid}).\n")
            f.write(f"holds({sid}, building).\n")
            f.write(f"holds({sid}, feature).\n")

            for col in df_out.columns:
                if col == "sample_id":
                    continue
                if row[col] == 1:
                    atom = col.lower()
                    f.write(f"holds({sid}, {atom}).\n")

            f.write("\n")


    print(f"Saved predictions to {output_csv_path} and {output_lp_path}")

    return logits_model, device, p



def run_clingo(lp_output, lp_to_run="utils/rules.lp"):
    #cmd = f"clingo utils/rules.lp {lp_output} --outf=2"
    result = subprocess.run(
    ["clingo", lp_to_run, lp_output, "--outf=2"],
    capture_output=True,
    text=True
)
    return result.stdout, result.stderr


def extract_rule_counts(clingo_output):
    #print(f"clingo out {clingo_output}")
    
    data = json.loads(clingo_output)

    #print(f"data {data}")

    rule_holds_counts = {}
    rule_vacuously_holds_count = {}
    violation_type_counts = defaultdict(dict)
    clingo_out_per_sample = defaultdict(dict)

    #print(f"data: {data}")

    for call in data["Call"]:
        for witness in call["Witnesses"]:
            for atom in witness["Value"]:

                if atom.startswith("rule_holds_count"):
                    # violation_count(rule, N)
                    inside = atom[len("rule_holds_count("):-1]
                    rule, n = inside.split(",")
                    rule_holds_counts[rule.strip()] = int(n.strip())

                elif atom.startswith("rule_vacuously_holds_count"):
                    # violation_count(rule, N)
                    inside = atom[len("rule_vacuously_holds_count("):-1]
                    rule, n = inside.split(",")
                    rule_vacuously_holds_count[rule.strip()] = int(n.strip())


                elif atom.startswith("violation_type_count"):
                    # violation_type_count(rule, type, N)
                    inside = atom[len("violation_type_count("):-1]
                    rule, vtype, n = inside.split(",")

                    rule = rule.strip()
                    vtype = vtype.strip()
                    n = int(n.strip())

                    violation_type_counts[rule][vtype] = n
                
                elif atom.startswith("rule_holds"):
                    inside = atom[len("rule_holds("):-1]
                    sample, rule_name = inside.split(",")
                    clingo_out_per_sample[sample.strip()][rule_name.strip()] = 1

                elif atom.startswith("rule_not_holds"):
                    inside = atom[len("rule_not_holds("):-1]
                    sample, rule_name, vtype = inside.split(",")
                    clingo_out_per_sample[sample.strip()][rule_name.strip()] = 0

                elif atom.startswith("rule_vacuously_holds"):
                    inside = atom[len("rule_vacuously_holds("):-1]
                    sample, rule_name = inside.split(",")
                    clingo_out_per_sample[sample.strip()][rule_name.strip()] = 1

    return rule_holds_counts, rule_vacuously_holds_count, violation_type_counts, clingo_out_per_sample


def compute_consistency_per_rule(clingo_output, txt_file):
    rule_holds_counts, rule_vacuously_holds_count, violation_type_counts, clingo_out_per_sample = extract_rule_counts(clingo_output)

    consistency = {}
    all_rules = set(rule_holds_counts.keys()) | set(violation_type_counts.keys()) | set(rule_vacuously_holds_count.keys())

    for rule in all_rules:
        holds = rule_holds_counts.get(rule, 0)
        violations = sum(violation_type_counts.get(rule, {}).values())
        #vac_holds = rule_vacuously_holds_count.get(rule, 0)
        total = holds + violations

        if total == 0:
            consistency[rule] = -2.0  # no samples → undefined
        else:
            consistency[rule] = holds / total  # fraction of times the rule is satisfied

    rules_and_concepts.print_only_active_rules(
        consistency, rule_holds_counts, rule_vacuously_holds_count, violation_type_counts, txt_file
    )

    return consistency, clingo_out_per_sample


def debug_batch_alignment(loader, logits_model, device):
    batch = next(iter(loader))
    features, labels, ids = batch
    features = features.to(device)
    with torch.no_grad():
        logits = logits_model(features)
    probs = torch.sigmoid(logits)
    print(f"\n[DEBUG] Sample: {ids[0]}")
    for i, cls in enumerate(CLASSES):
        label = labels[0, i].item()
        pred = probs[0, i].item()
        print(f"  {cls:20s}  label={int(label)}  pred={pred:.3f}")
    



'''OUTPUTS'''
def write_results_description_in_txt(txt_file, model, epochs_completed, warmup_epochs, final_epochs, final_alpha, starting_alpha, patience, constant_alpha, subset_class_pos_rates, info_about_dataset, seed):
    txt_file.write("Results Description\n")
    txt_file.write(f"Seed: {seed}")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Max nbr of epochs: %d\n" % EPOCHS)
    txt_file.write(f"Validation Interval: {VALIDATION_INTERVAL}")
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write(f"Number of final classes: {final_classes}\n" )
    txt_file.write(f"Number of concepts: {len(CLASSES) - final_classes}\n" )
    if TRAIN_WITH_A_SUBSET: txt_file.write(info_about_dataset)
    txt_file.write("\n")

    txt_file.write(f"Number of epochs completed: {epochs_completed}\n")
    txt_file.write(f"Warmup epochs: {warmup_epochs}; final_epochs: {final_epochs}; constant_alpha: {constant_alpha};  patience: {patience}\n")
    if not constant_alpha: txt_file.write(f"staring alpha:{starting_alpha}; final alpha: {final_alpha};\n")
    txt_file.write("\n")

    if model is not None:
        txt_file.write("\nModel Architecture:\n")
        txt_file.write(str(model))
        txt_file.write("\n\n")

    if starting_alpha < 1.0:
        rules_and_concepts.write_active_axioms(txt_file)

    if TRAIN_WITH_A_SUBSET:
        txt_file.write("\n\n\n\n")
        txt_file.write(f"[subset] class positive rates:\n")
        txt_file.write(subset_class_pos_rates)


def print_conf_matrices(conf_matrices):
    print("------------ FORMATO DA MATRIZ --------")
    print("\n")
    print("----------------[TN, FP]------------------")
    print("----------------[FN, TP]------------------")
    print("\n")

    print("TN: predicted Negative, Reality Negative")
    print("FP: predicted Positive, Reality Negative")
    print("FN: predicted Negative, Reality True")
    print("TP: predicted Positive, Reality Positive")

    print("\n")

    print("tn = torch.sum((y_true == 0) & (y_pred == 0)) #TN")
    print("fp = torch.sum((y_true == 0) & (y_pred == 1)) #FP")
    print("fn = torch.sum((y_true == 1) & (y_pred == 0)) #FN")
    print("tp = torch.sum((y_true == 1) & (y_pred == 1)) #TP")
    
        
    print("\n\n\n")

    for key, cm in conf_matrices.items():
        print(f"{key} confusion matrix:\n{cm}\n")

    


def write_conf_matrices_to_file(conf_matrices, txt_file):

    txt_file.write("------------ FORMATO DA MATRIZ --------")
    txt_file.write("\n")
    txt_file.write("----------------[TN, FP]------------------\n")
    txt_file.write("----------------[FN, TP]------------------\n")
    txt_file.write("\n")
    txt_file.write("TN: predicted Negative, Reality Negative\n")
    txt_file.write("FN: predicted Negative, Reality Positive\n")
    txt_file.write("FP: predicted Positive, Reality Negative\n")
    txt_file.write("TP: predicted Positive, Reality Positive\n")
    txt_file.write("\n")
    txt_file.write("------------ METRICAS --------")
    txt_file.write("\n")
    txt_file.write("precision: TP / (TP + FP) \n")
    txt_file.write("recall: TP / (TP + FN) \n")
    txt_file.write("f1: 2 * precision * recall / (precision + recall) \n")
    txt_file.write("\n\n\n")
    

    high_bias = {}
    high_bias_threshould, low_bias_threshould = 20, -20
    for key, cm in conf_matrices.items():
        print(f"{key} confusion matrix:\n{cm}\n")
        txt_file.write(f"{key} confusion matrix:\n{cm}\n")
        predicted_pos_rate, actual_pos_rate, bias, precision, recall, f1 = positive_rate_stats(cm)
        txt_file.write(f"predicted positive: {predicted_pos_rate:.3f}% | actual positive: {actual_pos_rate:.3f}% | bias: {bias:.3f}%\n")
        txt_file.write(f"precision: {precision:.3f}% | recall: {recall:.3f}% | f1: {f1:.3f}%\n\n\n")

        if bias > high_bias_threshould or bias < low_bias_threshould:
            high_bias[key] = cm

    txt_file.write("\n\n\n")
    txt_file.write("------------ HIGH BIAS METRICS --------\n")
    for key, cm in high_bias.items():
        txt_file.write(f"{key} confusion matrix:\n{cm}\n")
        predicted_pos_rate, actual_pos_rate, bias, precision, recall, f1 = positive_rate_stats(cm)
        txt_file.write(f"predicted positive: {predicted_pos_rate:.3f}% | actual positive: {actual_pos_rate:.3f}% | bias: {bias:.3f}%\n")
        txt_file.write(f"precision: {precision:.3f}% | recall: {recall:.3f}% | f1: {f1:.3f}%\n\n\n")


    

def positive_rate_stats(cm):
    tn, fp, fn, tp = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
    total = tn + fp + fn + tp
    predicted_positive_rate = (fp + tp) *100 / total
    actual_positive_rate = (fn + tp)*100 / total
    bias = predicted_positive_rate - actual_positive_rate

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall *100 / (precision + recall) if (precision + recall) else 0

    precision = precision *100
    recall = recall *100

    return predicted_positive_rate, actual_positive_rate, bias, precision, recall, f1

def summarize_last_epoch_sat_metrics(csv_path, txt_file, clingo_ltn_comparison, disagreement_values):
    df = pd.read_csv(csv_path)

    if df.empty:
        print("CSV is empty")
        return

    last = df.iloc[-1]

    # --- Core training info ---
    txt_file.write("\n========= Rule Satisfaction in LTNs =========\n")
    #txt_file.write("\n========= TODO: DISAGREEMENT VALUES =========\n")


    # --- Accuracy groups ---
    train_acc = {k.replace("_train_accuracy", ""): last[k]
                 for k in df.columns if k.endswith("_train_accuracy")}

    val_acc = {k.replace("_val_accuracy", ""): last[k]
               for k in df.columns if k.endswith("_val_accuracy")}

    
    #active_axioms = rules_and_concepts.get_active_axioms()

    txt_file.write("\nAxiom satisfaction: training\n\n")
    for k, v in sorted(train_acc.items(), key=lambda x: -x[1]):
        txt_file.write(f"{k:20s}: {v:.3f}\n")

    txt_file.write("\nAxiom satisfaction: validation \n\n")
    for k, v in sorted(val_acc.items(), key=lambda x: -x[1]):
        txt_file.write(f"{k:20s}: {v:.3f}\n")


    txt_file.write("\n" + "="*50)
    txt_file.write("\n========= LTN vs Clingo =========\n")

    txt_file.write("\n agreement per axiom \n\n")
    for k,v in sorted(val_acc.items(), key=lambda x: -x[1]):
        comparison = clingo_ltn_comparison.get(k, {})
        txt_file.write(f"{k:20s}: {comparison}\n")

    txt_file.write("\n\n Disagreements between LTN and CLINGO \n\n")
    for rule_name, sid_dict in disagreement_values.items():
        txt_file.write(f"\n\n\n--- {rule_name} ({len(sid_dict)} disagreements) ---\n")
        for sid, values in sid_dict.items():
            txt_file.write(f"  {sid}\n")
            txt_file.write(f"    ltn={values['ltn_binary']} clingo={values['clingo_binary']}"
                        f" body={values['final_result']:.4f}")
            
            if values['final_result'] > 0.6 or values['final_result'] < 0.4:
                txt_file.write("  PROBLEMATIC RESULT, check for weak antecedent pattern Reisenbarch\n")
            else:
                txt_file.write("\n")


            for cn, pv in values["concepts"].items():
                distance = abs(pv - 0.5)
                txt_file.write(f"    {cn:15s} = {pv:.4f}  (|d-0.5|={distance:.4f})\n")



def summarize_last_epoch(csv_path, txt_file):
    df = pd.read_csv(csv_path)

    if df.empty:
        print("CSV is empty")
        return

    last = df.iloc[-1]

    txt_file.write("\n" + "="*50)
    txt_file.write(f"FINAL EPOCH SUMMARY (Epoch {int(last['Epoch'])})")
    txt_file.write("="*50)

    # --- Core training info ---
    txt_file.write("\nTraining Info\n")
    txt_file.write(f"Alpha: {last['Alpha']:.4f}\n")
    txt_file.write(f"Classification Train Loss: {last['train_loss']:.4f}\n")
    txt_file.write(f"Classification Val Loss: {last['val_loss']:.4f}\n")
    txt_file.write(f"Combined Train Loss: {last['combined_train_loss']:.4f}\n")
    txt_file.write(f"Combined Val Loss: {last['combined_val_loss']:.4f}\n")

    # --- Logic consistency ---
    txt_file.write("\nLogical Consistency\n")
    txt_file.write(f"Train SAT (KB): {last['train_sat_kb']:.4f}\n")
    txt_file.write(f"Val SAT (KB): {last['val_sat_kb']:.4f}\n")

    # --- Accuracy groups ---
    train_acc = {k.replace("_train_accuracy", ""): last[k]
                 for k in df.columns if k.endswith("_train_accuracy")}

    val_acc = {k.replace("_val_accuracy", ""): last[k]
               for k in df.columns if k.endswith("_val_accuracy")}

    
    final_classes = rules_and_concepts.get_normalized_FINAL_CLASSES()

    class_train_acc = {k: train_acc[k] for k in train_acc if k in final_classes}
    concept_train_acc = {k: train_acc[k] for k in train_acc if k not in final_classes}
    class_val_ac = {k: val_acc[k] for k in val_acc if k in final_classes}
    concept_val_acc = {k: val_acc[k] for k in val_acc if k not in final_classes}



    txt_file.write("\nTrain Accuracy per Class, final classes\n")
    for k, v in sorted(class_train_acc.items(), key=lambda x: -x[1]):
        txt_file.write(f"{k:20s}: {v:.3f}\n")

    txt_file.write("\nTrain Accuracy per Class, concepts\n")
    for k, v in sorted(concept_train_acc.items(), key=lambda x: -x[1]):
        txt_file.write(f"{k:20s}: {v:.3f}\n")

    txt_file.write("\nValidation Accuracy per Class, final classes\n")
    for k, v in sorted(class_val_ac.items(), key=lambda x: -x[1]):
        txt_file.write(f"{k:20s}: {v:.3f}\n")

    txt_file.write("\nValidation Accuracy per Class, concepts\n")
    for k, v in sorted(concept_val_acc.items(), key=lambda x: -x[1]):
        txt_file.write(f"{k:20s}: {v:.3f}\n")

    txt_file.write("\n" + "="*50)

#print_conf_matrices(conf_matrices)


#final_epochs: epochs with final alpha
def new_calculate_alpha(epoch= 0, warmup_epochs = 10, final_alpha = 0.25, final_epochs=30, starting_alpha=1.0):
    alpha = starting_alpha


    if epoch < warmup_epochs:
        alpha = starting_alpha
    elif epoch < EPOCHS - final_epochs:
        progress = (epoch - warmup_epochs) / (EPOCHS - warmup_epochs-final_epochs)
        alpha = starting_alpha - progress * (starting_alpha - final_alpha)
    else:
        alpha = final_alpha

    return alpha, False


def evaluate_conf_matrix(model_path, learning_rate, alpha):
    logits_model, device, classification_loss_fn, p = reset_model(in_channels=3, num_classes=final_classes)

    logits_model.load_state_dict(torch.load(model_path))
    logits_model.to(device)
    logits_model.eval()

    reset_conf_matrices(conf_matrices)
    optimizer = torch.optim.Adam(logits_model.parameters(), lr=learning_rate)

    for batch_elements in train_loader:
        features_param, labels_list = __get_elements_from_batch(batch_elements)

        # Move to device if needed
        features_param = features_param.to(device)
        labels_list = [lbl.to(device) for lbl in labels_list]

        train_step(features_param, *labels_list, optimizer=optimizer, logits_model=logits_model, classification_loss_fn=classification_loss_fn, p=p, alpha=alpha)


    for batch_elements_test in test_loader:
        features_param, labels_list = __get_elements_from_batch(batch_elements_test)

        # Move to device if needed
        features_param = features_param.to(device)
        labels_list = [lbl.to(device) for lbl in labels_list]

        test_step(features_param, *labels_list, logits_model=logits_model, classification_loss_fn=classification_loss_fn, p=p, alpha=alpha)

    
    print_conf_matrices(conf_matrices)

def run_consistency_validation(model_path, output_csv_path, output_lp_path, txt_file, lp_to_run="rules.lp", sample_loader=test_loader, debug_mode=False):
    
    model_mode= False #CNN model in eval mode. if its in train mode, the dropout layers mess up the results bc they have different values

    print("\n generating clingo rules \n")
    new_lp_file = open(lp_to_run, "w+")
    test_new_rules.generate_clingo_rules(rules_and_concepts.get_all_axioms(), new_lp_file, debug_mode=debug_mode)
    new_lp_file.close()


    print("\n starting testing \n")
    logits_model, device, p = export_predictions_to_csv_and_lp(model_path, output_csv_path, output_lp_path, sample_loader=sample_loader)

    clingo_output, clingo_stederr = run_clingo(output_lp_path, lp_to_run=lp_to_run)
    print(f" clingo_stderr {clingo_stederr}")
    output, clingo_output_per_sample = compute_consistency_per_rule(clingo_output, txt_file)

    ltn_output_per_sample, comparison_per_rule, disagreement_values = axioms.evaluate_per_sample_axioms(logits_model, sample_loader, p, device, clingo_output_per_sample)


    #comparison_per_rule = compare_ltn_vs_clingo(ltn_output_per_sample, clingo_output_per_sample)

    print(f"clingo ou per sample {clingo_output_per_sample}\n")
    print(f"consistency results: {output}\n" )
    print(f"disagreeement values: {disagreement_values}\n")

    return clingo_output_per_sample, ltn_output_per_sample, comparison_per_rule, disagreement_values


def compare_ltn_vs_clingo(ltn_per_sample, clingo_per_sample, raw_ltn_output_per_sample, x):
    per_rule = defaultdict(lambda: {"agree": 0, "disagree": 0, "missing": 0})
    #raw_axiom_values = defaultdict(lambda: {"concepts": {}, "final_result": 0.0})

    disagreement_values = {} #key = rule_name
    for sid, ltn_rules in ltn_per_sample.items():
        
        clingo_rules = clingo_per_sample.get(sid, {})
        rule_disagreements = {} #key = sid

        for rule_name, ltn_val in ltn_rules.items():
            if rule_name not in clingo_rules:
                per_rule[rule_name]["missing"] += 1
                continue
            if ltn_val == clingo_rules[rule_name]:
                per_rule[rule_name]["agree"] += 1
            else:
                per_rule[rule_name]["disagree"] += 1
                raw_axiom_values = axioms.recover_axiom_values(rule_name, sid, raw_ltn_output_per_sample, x)
                rule_disagreements[sid] = raw_axiom_values
        
        disagreement_values[rule_name] = rule_disagreements


    return per_rule, disagreement_values

def write_to_csv_file(template, csv_template, results, epoch, alpha, csv_file):

    #metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *results))
    if csv_path is not None:
        #csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.write(csv_template.format(epoch, alpha, *results) + "\n")
        csv_file.flush()

    


""" SCRIPT """
LIMIT = 150000
EPOCHS = 120 #isto serve para definir quao gradualmente o alpha é modeficado, 150 era fixe para o conhecimento original

#EARLY_STOPPING_DELTA = 0.005
PATIENCE = 15
patience_multiplier = 1
if TRAIN_WITH_A_SUBSET: 
    patience_multiplier = LEN_TRAIN_DATASET / SUBSET_SIZE
    print(f"FINAL PATIENCE: {PATIENCE*patience_multiplier}") 


MODEL_SAVING_INTERVAL = 100
LEARNING_RATE=0.0001
output_dir = ""

#metrics_dict.update(class_train_accuracy_metric)

def train_loop(warmup_epochs = 10, final_alpha = 0.25, final_epochs=30, starting_alpha=1.0, patience=20, early_stopping_delta=0.001, learning_rate=0.001, train_model_number=0, seed=None):
    #reset model
    logits_model, device, classification_loss_fn, p = reset_model(in_channels=3, num_classes=final_classes, seed=seed)
    debug_batch_alignment(train_loader, logits_model, device)

    #folder
    run_name = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    output_dir = Path(f"{run_name}_outputs{train_model_number}")
    output_dir = "outputs" / output_dir
    output_dir.mkdir(parents=True, exist_ok=True)

    csv_path = output_dir / "metrics.csv"
    satAxiom_csv_path = output_dir / "sat_level_per_axiom.csv"
    predictions_csv_path = output_dir / "raw_predictions_val.csv"
    predictions_csv_path_train = output_dir / "raw_predictions_train.csv"
    output_lp_path = output_dir / "predictions_val.lp"
    output_lp_path_train = output_dir / "predictions_train.lp"
    txt_path = output_dir / "stats.txt"
    consistency_validation_path = output_dir / "consistency_validation_val.txt"
    consistency_validation_path_train = output_dir / "consistency_validation_train.txt"
    conf_matrix_path = output_dir / "conf_matrix.txt"
    txt_path_final_results = output_dir / "last_epoch_results.txt"
    patience_tracker_path = output_dir / "patience_tracker.txt"


    optimizer = torch.optim.Adam(logits_model.parameters(), lr=learning_rate)

    best_loss = 10000.0
    epochs_no_improve = 0

    epochs_completed = 0
    constant_alpha = True

    # scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
    template = "Epoch {}"
    satAxiom_template = "Epoch {}"

    satAxiom_keys = axioms.get_rule_sat_metrics_keys()

    for metrics_label in metrics_dict.keys():
        template += ", %s: {:.5f}" % metrics_label

    if csv_path is not None:

        csv_file = open(csv_path, "w+")
        headers = ",".join(["Epoch", "Alpha"] + list(metrics_dict.keys())) #sem o alpha
        csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 2)]) #era +1
        csv_file.write(headers + "\n")


    for metrics_label in satAxiom_keys:
        satAxiom_template += ", %s: {:.5f}" % metrics_label

    if satAxiom_csv_path is not None:

        satAxiom_csv_file = open(satAxiom_csv_path, "w+")
        satAxiom_headers = ",".join(["Epoch", "Alpha"] + list(satAxiom_keys)) #sem o alpha
        satAxiom_csv_template = ",".join(["{}" for _ in range(axioms.get_rule_sat_metrics_len() + 2)]) #era +1
        satAxiom_csv_file.write(satAxiom_headers + "\n")

    

    for epoch in range(LIMIT):
        print(f"starting epoch {epoch}")
        reset_conf_matrices(conf_matrices)
        axioms.reset_rule_sat_metrics()
        #reset metrics
        for metrics in metrics_dict.values():
            metrics.reset_state()

        #alpha, constant_alpha = new_calculate_alpha(epoch, warmup_epochs, final_alpha, final_epochs, starting_alpha)
        alpha = starting_alpha
        #alpha = 0.5
        metrics_dict["alpha"].update(alpha)

        print(f"alpha: {alpha}")

        for batch_elements in train_loader:
            features_param, labels_list = __get_elements_from_batch(batch_elements)

            #print("labels list")
            #print(labels_list.shape)

            # Move to device if needed
            features_param = features_param.to(device)
            labels_list = [lbl.to(device) for lbl in labels_list]

            #print("train step")
            train_step(features_param, *labels_list, optimizer=optimizer, logits_model=logits_model, classification_loss_fn=classification_loss_fn, p=p, alpha=alpha)

        if epoch % VALIDATION_INTERVAL == 0:

            for batch_elements_test in test_loader:
                features_param, labels_list = __get_elements_from_batch(batch_elements_test)

                # Move to device if needed
                features_param = features_param.to(device)
                labels_list = [lbl.to(device) for lbl in labels_list]

                test_step(features_param, *labels_list, logits_model=logits_model, classification_loss_fn=classification_loss_fn, p=p, alpha=alpha)


            metrics_results = [metrics.result() for metrics in metrics_dict.values()]
            write_to_csv_file(template, csv_template, metrics_results, epoch, alpha, csv_file)


            satAxiom_results = axioms.get_rule_sat_metrics_results()
            write_to_csv_file(satAxiom_template, satAxiom_csv_template, satAxiom_results, epoch, alpha, satAxiom_csv_file)


            #early stopping condition
            current_loss = metrics_dict["combined_val_loss"].result()
            new_loss = best_loss - early_stopping_delta #preciso de mudar este nome

            print(f"best_loss: {best_loss}, current_loss: {current_loss}, number of no improve runs:{epochs_no_improve}/{patience}")
            patience_tracker_file = open(patience_tracker_path, "w+")
            patience_tracker_file.write(f"best_loss: {best_loss}, current_loss: {current_loss}, number of no improve cycles of {VALIDATION_INTERVAL} epochs:{epochs_no_improve}/{patience}\n")
            patience_tracker_file.close()

            if current_loss < new_loss:
                best_loss = current_loss
                epochs_no_improve = 0
                print("\n\n\n CHANGED BEST LOSS \n\n\n")
            elif epoch > 0:
                #print("\n\n\n CHANGED BEST SAT \n\n\n")
                epochs_no_improve += 1


            if epochs_no_improve >= patience:
                print("EARLY STOPPING TRIGGERED")
                epochs_completed = epoch
                break

        #model saving
        if epoch != 0 and epoch % MODEL_SAVING_INTERVAL == 0:
            print(f"SAVING MODEL AT EPOCH {epoch}")
            torch.save(logits_model.state_dict(), output_dir / f"model_epoch_{epoch}.pth")


        #metrics_dict.clear()
        


    print("training finished, starting to save the model")
    txt_file = open(txt_path, "w+")
    txt_file_final_results = open(txt_path_final_results, "w+")
    conf_matrix_file = open(conf_matrix_path, "w+")
    write_results_description_in_txt(txt_file, logits_model, epochs_completed,warmup_epochs, final_epochs, final_alpha, starting_alpha, patience, constant_alpha, subset_class_pos_rates, info_about_dataset, seed)
    #txt_file.write("\n\n\n\n")
    
    #txt_file.write(subset_class_pos_rates)

    write_conf_matrices_to_file(conf_matrices, conf_matrix_file)

    summarize_last_epoch(csv_path, txt_file_final_results)

    model_path = output_dir / f"model_epoch_{epoch}.pth"

    torch.save(logits_model.state_dict(), model_path)

    consistency_validation_file = open(consistency_validation_path, "w+")
    consistency_validation_file_train = open(consistency_validation_path_train, "w+")

    _, _, clingo_ltn_comparison, disagreement_values = run_consistency_validation(model_path, predictions_csv_path, output_lp_path, consistency_validation_file, lp_to_run="utils/rules.lp", sample_loader=test_loader)
    summarize_last_epoch_sat_metrics(satAxiom_csv_path, consistency_validation_file, clingo_ltn_comparison, disagreement_values)

    _, _, clingo_ltn_comparison_train, disagreement_values_train = run_consistency_validation(model_path, predictions_csv_path_train, output_lp_path_train, consistency_validation_file_train, lp_to_run="utils/rules.lp", sample_loader=train_loader)
    summarize_last_epoch_sat_metrics(satAxiom_csv_path, consistency_validation_file_train, clingo_ltn_comparison_train, disagreement_values_train)

    if csv_path is not None:
        csv_file.close()
        satAxiom_csv_file.close()

        txt_file.close()
        txt_file_final_results.close()
        conf_matrix_file.close()
        consistency_validation_file.close()

    #default 0.7, 0.8, 0.9
for i in range(0,4):
    seed = torch.randint(0, 100000, (1,)).item()
    print(f"Run {i}, seed: {seed}")
    #train_loop(..., seed=seed, train_model_number=i)

    starting_alpha = 1.0
    if i== 0:
        starting_alpha = 1.0
    if i ==1:
        starting_alpha = 0.7
    if i ==2:
        starting_alpha = 0.5
    if i ==3:
        starting_alpha = 0.2

    patience = PATIENCE*patience_multiplier/VALIDATION_INTERVAL

    train_loop(warmup_epochs = 10, final_alpha = 0.25, final_epochs=30, starting_alpha= starting_alpha, patience=patience, early_stopping_delta=0.005, learning_rate=LEARNING_RATE, train_model_number=i, seed=seed)





    

Starting execution using DEBUG_MODE: False
Starting execution using TRAIN_WITH_A_SUBSET: True
SUBSET SIZE: 12000, SUBSET_SEED: 42, STRATIFY:True 
[DEBUG axioms] CLASS_TO_IDX: {'Cafe': 0, 'Hotel': 1, 'Restaurant': 2, 'Store': 3, 'MiscCommercial': 4, 'Suburban': 5, 'MiscResidential': 6, 'CountryHouse': 7, 'ConstructionSite': 8, 'MiscIndustrial': 9, 'PowerPlant': 10, 'WaterTreatment': 11, 'Residential': 12, 'Commercial': 13, 'Industrial': 14, 'Door': 15, 'Window': 16, 'Awning': 17, 'Billboard': 18, 'Porch': 19, 'Sign': 20, 'Table': 21, 'TiledRoof': 22, 'TiledRoofTop': 23, 'VendingMachine': 24, 'WallSign': 25, 'Statue': 26, 'Chimney': 27, 'Pipe': 28, 'Machine': 29, 'Truck': 30, 'Car': 31}
[DEBUG] CLASSES order: ['Cafe', 'Hotel', 'Restaurant', 'Store', 'MiscCommercial', 'Suburban', 'MiscResidential', 'CountryHouse', 'ConstructionSite', 'MiscIndustrial', 'PowerPlant', 'WaterTreatment', 'Residential', 'Commercial', 'Industrial', 'Door', 'Window', 'Awning', 'Billboard', 'Porch', 'Sign', 'Table

In [14]:

FEATURE_CLASSES = rules_and_concepts.get_concepts()

def generate_INDIVIDUALS():
    indv = []

    for f in FEATURE_CLASSES:
        f_tuple = (f.lower(), f)

        indv.append(f_tuple)

    return indv


INDIVIDUALS = generate_INDIVIDUALS()
print(INDIVIDUALS)

[('door', 'Door'), ('awning', 'Awning'), ('billboard', 'Billboard'), ('porch', 'Porch'), ('tiledroof', 'TiledRoof'), ('vendingmachine', 'VendingMachine'), ('wallsign', 'WallSign'), ('pipe', 'Pipe'), ('machine', 'Machine'), ('car', 'Car')]


In [13]:
conf_matrix_file = open("conf_test.txt", "w+")
write_conf_matrices_to_file(conf_matrices, conf_matrix_file)

conf_matrix_file.close()

Hotel_train confusion matrix:
tensor([[ 0, 26],
        [ 0,  6]], dtype=torch.int32)

Store_train confusion matrix:
tensor([[ 0, 28],
        [ 0,  4]], dtype=torch.int32)

Suburban_train confusion matrix:
tensor([[ 0, 20],
        [ 0, 12]], dtype=torch.int32)

ConstructionSite_train confusion matrix:
tensor([[ 0, 30],
        [ 0,  2]], dtype=torch.int32)

Awning_train confusion matrix:
tensor([[16,  0],
        [16,  0]], dtype=torch.int32)

Billboard_train confusion matrix:
tensor([[ 0, 28],
        [ 0,  4]], dtype=torch.int32)

Porch_train confusion matrix:
tensor([[ 0, 20],
        [ 0, 12]], dtype=torch.int32)

TiledRoof_train confusion matrix:
tensor([[ 0, 14],
        [ 0, 18]], dtype=torch.int32)

WallSign_train confusion matrix:
tensor([[26,  0],
        [ 6,  0]], dtype=torch.int32)

Pipe_train confusion matrix:
tensor([[ 0, 29],
        [ 0,  3]], dtype=torch.int32)

Machine_train confusion matrix:
tensor([[ 0, 30],
        [ 0,  2]], dtype=torch.int32)

Car_train confus

In [6]:
import utils.axioms as axioms
import utils.occlusion as occlusion
import utils.rules_and_concepts as rules_and_concepts
import utils.test_new_rules as test_new_rules
#import charter
importlib.reload(axioms)
importlib.reload(occlusion)
importlib.reload(rules_and_concepts)
importlib.reload(test_new_rules)

out_dir = "2026-06-01_03-22-09_outputs0"
epoch_nbr = 15

model_path = f"outputs/{out_dir}/model_epoch_{epoch_nbr}.pth"
satAxiom_csv_path = f"outputs/{out_dir}/sat_level_per_axiom.csv"

output_csv_path = f"outputs/{out_dir}/results_for_validation_test.csv"
output_lp_path = f"outputs/{out_dir}/predictions_test.lp"
txt_file = open("utils/test3_eval.txt", "w+")

output_csv_path_train = f"outputs/{out_dir}/results_for_validation_train.csv"
output_lp_path_train = f"outputs/{out_dir}/predictions_train.lp"
txt_file_train = open("utils/test3_train.txt", "w+")

clingo_output, ltn_output, comparison, disagreement_values = run_consistency_validation(model_path, output_csv_path, output_lp_path, txt_file, lp_to_run="utils/rules.lp", sample_loader=test_loader)
summarize_last_epoch_sat_metrics(satAxiom_csv_path, txt_file, comparison, disagreement_values)

clingo_output_train, ltn_output_train, comparison_train, disagreement_values_train = run_consistency_validation(model_path, output_csv_path_train, output_lp_path_train, txt_file_train, lp_to_run="utils/rules.lp", sample_loader=train_loader)
summarize_last_epoch_sat_metrics(satAxiom_csv_path, txt_file_train, comparison_train, disagreement_values_train)

print(f"clingo output {clingo_output}\n")
#print(f"ltn rules {lt_output}\n")
#print(f"compared = {comparison}\n")
#print(f"disagreeement_balues: {disagreement_values}")

txt_file.close()
txt_file_train.close()

classes in axioms.py ['Cafe', 'Hotel', 'Restaurant', 'Store', 'MiscCommercial', 'Door', 'Awning', 'TiledRoof', 'VendingMachine', 'Pipe', 'Car']
final classes  in rules and concepts['Cafe', 'Hotel', 'Restaurant', 'Store', 'MiscCommercial']
final concepts ['Door', 'Awning', 'TiledRoof', 'VendingMachine', 'Pipe', 'Car']
classes in concepts and rules ['Cafe', 'Hotel', 'Restaurant', 'Store', 'MiscCommercial', 'Door', 'Awning', 'TiledRoof', 'VendingMachine', 'Pipe', 'Car']
left: ['Cafe'] | right: [['NOT(Car)', 'Awning']]
in clingo holds left: [['Cafe']], right: [['NOT(Car)', 'Awning']]
rule in part_holds ['Cafe'], in part left
rule in part_holds ['NOT(Car)', 'Awning'], in part right
main op in gen stuff MainOperations.EQUIVALENCE







 inside condition EQUIV 







left: ['Hotel'] | right: [['Car', 'Pipe']]
in clingo holds left: [['Hotel']], right: [['Car', 'Pipe']]
rule in part_holds ['Hotel'], in part left
rule in part_holds ['Car', 'Pipe'], in part right
main op in gen stuff MainOperat

In [8]:
txt_file_train.close()

JSON CONFIG for the dictionary

In [5]:
import utils.axioms as axioms
import utils.occlusion as occlusion
import utils.rules_and_concepts as rules_and_concepts
#import charter
importlib.reload(axioms)
importlib.reload(occlusion)
importlib.reload(rules_and_concepts)

rules_and_concepts.generate_json_config_file()

classes in axioms.py ['Hotel', 'Store', 'Suburban', 'ConstructionSite', 'Door', 'Window', 'Awning', 'Billboard', 'Porch', 'Table', 'WallSign', 'Chimney', 'Pipe', 'Machine', 'Truck', 'Car']
rule in getClassesFromRules ['Hotel', 'WallSign']
rule in getClassesFromRules ['Store', 'Billboard']
rule in getClassesFromRules ['ConstructionSite', 'Machine']
rule in getClassesFromRules ['Suburban', 'Porch']
rule in getClassesFromRules ['Hotel', 'AND', 'Door', ' Chimney', '']
rule in getClassesFromRules ['Hotel', 'AND', 'Window', ' Pipe', '']
rule in getClassesFromRules ['Hotel', 'AND', 'Car', ' Awning', '']
rule in getClassesFromRules ['Store', 'AND', 'Door', ' NOT', 'Chimney', '', '']
rule in getClassesFromRules ['Store', 'AND', 'Window', ' NOT', 'Pipe', '', '']
rule in getClassesFromRules ['Store', 'AND', 'Car', ' NOT', 'Awning', '', '']
rule in getClassesFromRules ['ConstructionSite', 'AND', 'Truck', ' Chimney', '']
rule in getClassesFromRules ['ConstructionSite', 'AND', 'Table', ' Pipe', '']


In [63]:
import utils.axioms as axioms
import utils.occlusion as occlusion
import utils.rules_and_concepts as rules_and_concepts
#import charter
importlib.reload(axioms)
importlib.reload(occlusion)
importlib.reload(rules_and_concepts)

out_path = "2026-04-27_14-34-54_outputs0_IndAndRes"

csv_path = f"outputs/{out_path}/metrics.csv"
txt_file_test = open(f"outputs/{out_path}/last_epoch_results.txt", "w+")

def summarize_last_epoch(csv_path, txt_file):
    df = pd.read_csv(csv_path)

    if df.empty:
        print("CSV is empty")
        return

    last = df.iloc[-1]

    txt_file.write("\n" + "="*50)
    txt_file.write(f"FINAL EPOCH SUMMARY (Epoch {int(last['Epoch'])})")
    txt_file.write("="*50)

    # --- Core training info ---
    txt_file.write("\nTraining Info\n")
    txt_file.write(f"Alpha: {last['Alpha']:.4f}\n")
    txt_file.write(f"Train Loss: {last['train_loss']:.8f}\n")
    txt_file.write(f"Val Loss: {last['val_loss']:.8f}\n")

    # --- Logic consistency ---
    txt_file.write("\nLogical Consistency\n")
    txt_file.write(f"Train SAT (KB): {last['train_sat_kb']:.8f}\n")
    txt_file.write(f"Val SAT (KB): {last['val_sat_kb']:.8f}\n")

    txt_file.write("\nCombined Loss\n")
    txt_file.write(f"Combined Train Loss: {last['combined_train_loss']:.8f}\n")
    txt_file.write(f"Combined Val Loss: {last['combined_val_loss']:.8}\n")


    # --- Accuracy groups ---
    train_acc = {k.replace("_train_accuracy", ""): last[k]
                 for k in df.columns if k.endswith("_train_accuracy")}

    val_acc = {k.replace("_val_accuracy", ""): last[k]
               for k in df.columns if k.endswith("_val_accuracy")}

    
    final_classes = rules_and_concepts.get_normalized_FINAL_CLASSES()

    class_train_acc = {k: train_acc[k] for k in train_acc if k in final_classes}
    concept_train_acc = {k: train_acc[k] for k in train_acc if k not in final_classes}
    class_val_ac = {k: val_acc[k] for k in val_acc if k in final_classes}
    concept_val_acc = {k: val_acc[k] for k in val_acc if k not in final_classes}



    txt_file.write("\nTrain Accuracy per Class, final classes\n")
    for k, v in sorted(class_train_acc.items(), key=lambda x: -x[1]):
        txt_file.write(f"{k:20s}: {v:.3f}\n")

    txt_file.write("\nTrain Accuracy per Class, concepts\n")
    for k, v in sorted(concept_train_acc.items(), key=lambda x: -x[1]):
        txt_file.write(f"{k:20s}: {v:.3f}\n")

    txt_file.write("\nValidation Accuracy per Class, final classes\n")
    for k, v in sorted(class_val_ac.items(), key=lambda x: -x[1]):
        txt_file.write(f"{k:20s}: {v:.3f}\n")

    txt_file.write("\nValidation Accuracy per Class, concepts\n")
    for k, v in sorted(concept_val_acc.items(), key=lambda x: -x[1]):
        txt_file.write(f"{k:20s}: {v:.3f}\n")

    txt_file.write("\n" + "="*50)
    txt_file_test.close()


summarize_last_epoch(csv_path, txt_file_test)

classes ['Suburban', 'MiscResidential', 'CountryHouse', 'ConstructionSite', 'MiscIndustrial', 'PowerPlant', 'WaterTreatment', 'Residential', 'Industrial', 'Awning', 'Porch', 'Table', 'TiledRoof', 'Chimney', 'Pipe', 'Machine', 'Truck', 'Car']
rule in getClassesFromRules ['Industrial', 'OR', 'ConstructionSite', ' MiscIndustrial', ' PowerPlant', ' WaterTreatment', '']
rule in getClassesFromRules ['Residential', 'OR', 'CountryHouse', ' MiscResidential', ' Suburban', '']
rule in getClassesFromRules ['Industrial', 'NOT', 'Table', '']
rule in getClassesFromRules ['ConstructionSite', 'Machine']
rule in getClassesFromRules ['Suburban', 'Porch']
rule in getClassesFromRules ['MiscIndustrial', 'AND', 'NOT', 'Awning', '', ' Truck', '']
rule in getClassesFromRules ['CountryHouse', 'AND', 'Car', ' TiledRoof', '']
rule in getClassesFromRules ['PowerPlant', 'AND', 'Chimney', ' Pipe', '']
rule in getClassesFromRules ['WaterTreatment', 'AND', 'Pipe', ' Truck', '']
rule in getClassesFromRules ['MiscReside

Occlusion map

In [30]:
import utils.axioms as axioms
import utils.occlusion as occlusion
importlib.reload(axioms)
importlib.reload(occlusion)

ALL_EXISTING_CLASSES = axioms.get_ALL_CLASSES()

output_dir = "outputs/2026-04-08_01-49-56_outputs0"
model_path = "outputs/2026-04-08_01-49-56_outputs0/model_epoch_31.pth"
txt_path = "outputs/2026-04-08_01-49-56_outputs0/plots/images/images_info.txt"

logits_model = CNN(in_channels=3, num_classes=1)#3, final classes

logits_model.load_state_dict(torch.load(model_path))

for i in range(3):
    id = i +10
    img, target = test_dataset[id]


    heatmap = occlusion.occlusion_map(logits_model, img, class_idx=0)
    occlusion.show_occlusion(img, heatmap, output_dir, id=id)

    occlusion.write_image_info(relevant_classes=CLASSES, all_classes=ALL_EXISTING_CLASSES,image_id=i, image_concepts=target, txt_path=txt_path)

size of CNN
20000
relevant concepts: ['Table'] 
all concepts: ['Industrial']
relevant concepts: ['MiscCommercial', 'Awning', 'Table'] 
all concepts: ['Residential', 'Commercial', 'Industrial']
relevant concepts: [] 
all concepts: []


In [ ]:
epoch = 57
torch.save(logits_model.state_dict(), "model_epoch_57.pth")

In [68]:
'''
        # Compute confusion matrix manually
        tn = torch.sum((y_true == 0) & (y_pred == 0)) #TN
        fp = torch.sum((y_true == 0) & (y_pred == 1)) #FP
        fn = torch.sum((y_true == 1) & (y_pred == 0)) #FN
        tp = torch.sum((y_true == 1) & (y_pred == 1)) 
'''

model_path = "outputs/2026-04-13_03-08-32_outputs2/model_epoch_46.pth"

#tem de ser a mao
'''

NAO ESQUECER DE MUDAR O ALPHA

'''
alpha = 0.9
'''

NAO ESQUECER DE MUDAR O ALPHA

'''


evaluate_conf_matrix(model_path, LEARNING_RATE, alpha=alpha)

size of CNN
20000
Sat Function Result: 0.9965331554412842, Class_loss: 1.456361587770516e-05, ltn-loss: 0.0034668445587158203, Combined_loss:0.00035979170934297144 
Sat Function Result: 0.9889622926712036, Class_loss: 0.0008558517438359559, ltn-loss: 0.011037707328796387, Combined_loss:0.0018740373197942972 
Sat Function Result: 0.964794933795929, Class_loss: 0.008838444016873837, ltn-loss: 0.035205066204071045, Combined_loss:0.01147510576993227 
Sat Function Result: 0.9906336665153503, Class_loss: 0.0001849812688305974, ltn-loss: 0.009366333484649658, Combined_loss:0.0011031165486201644 
Sat Function Result: 0.993842601776123, Class_loss: 3.546455263858661e-05, ltn-loss: 0.006157398223876953, Combined_loss:0.0006476579583249986 
Sat Function Result: 0.9859523177146912, Class_loss: 0.000616523262578994, ltn-loss: 0.014047682285308838, Combined_loss:0.001959639135748148 
Sat Function Result: 0.9897928833961487, Class_loss: 9.707509161671624e-05, ltn-loss: 0.010207116603851318, Combined_

old code

In [ ]:

data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(DATA_ROOT, "all.csv")
images = Path(DATA_ROOT, "c1")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]
CLASSES = [
    "Cafe",
    "Hotel",
    "Store",
    "MiscCommercial",
    "Awning",
    "Billboard",
    "VendingMachine",
    "Statue",
    "Table",
    "WallSign",
]
final_classes = 4

class BuildingsDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = self.img_dir / row["id"]
        image = Image.open(img_path).convert("RGB")



        #target = torch.tensor(row[CLASSES].values, dtype=torch.float32)
        target = torch.tensor(row[CLASSES].to_numpy(dtype=float), dtype=torch.float32)

        #print(row[CLASSES].to_numpy().dtype)


        if self.transform:
            image = self.transform(image)

        return image, target


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


TRAIN_TEST_LIMIT = int(0.8 * len(df))

train_df = df[:TRAIN_TEST_LIMIT]
test_df = df[TRAIN_TEST_LIMIT:]

df[CLASSES] = df[CLASSES].apply(pd.to_numeric)


train_dataset = BuildingsDataset(train_df, images, transform=transform)
test_dataset = BuildingsDataset(test_df, images, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

#print(train_df["Cafe"])

"""CNN MODEL"""

class CNN(nn.Module):
   def __init__(self, in_channels, num_classes, name="cnn_model"):

       """
       Building blocks of convolutional neural network.

       Parameters:
           * in_channels: Number of channels in the input image (for grayscale images, 1)
           * num_classes: Number of classes to predict. In our problem, 10 (i.e digits from  0 to 9).
       """
       super(CNN, self).__init__()

       #layers
       self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3)
       self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3)
       self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
       #self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3)

       self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
       self.flatten = nn.Flatten()
       self.dropout = nn.Dropout(0.5)

        #3 conv 43624, 4 conv 18432
       self.dense1 = nn.Linear(43264, 128)#com 3 conv, 43264
       self.dense2 = nn.Linear(128, 64)

       self.concept_layer = nn.Linear(64, 32)
       self.output_layer = nn.Linear(32, num_classes)


   def forward(self, inputs):
       """
       Define the forward pass of the neural network.

       Parameters:
           x: Input tensor.

       Returns:
           torch.Tensor
               The output tensor after passing through the network.
       """
       if isinstance(inputs, (list, tuple)):
          inputs = inputs[0]


       x = F.relu(self.conv1(inputs))
       x = self.maxpool(x)
       x = F.relu(self.conv2(x))
       x = self.maxpool(x)
       x = F.relu(self.conv3(x))
       x = self.maxpool(x)

       x = self.flatten(x)
       x = self.dropout(x)

       x = F.relu(self.dense1(x))
       x = self.dropout(x)
       x = F.relu(self.dense2(x))

       y = F.relu(self.concept_layer(x))
       x = self.output_layer(y)

       result = torch.cat([x, y], dim=1)
       return result

      
'''
class CNN(nn.Module):
    def __init__(self, in_channels, num_classes, name="cnn_model"):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, 3)
        self.conv2 = nn.Conv2d(16, 32, 3)
        self.conv3 = nn.Conv2d(32, 64, 3)

        self.pool = nn.MaxPool2d(2)

        self.gap = nn.AdaptiveAvgPool2d((1,1))

        self.fc1 = nn.Linear(64, 128)
        self.fc2 = nn.Linear(128, 64)

        self.concept_layer = nn.Linear(64, 32)
        self.output_layer = nn.Linear(32, num_classes)

    def forward(self, x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = self.gap(x)
        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        y = F.relu(self.concept_layer(x))
        x = self.output_layer(y)

        return torch.cat([x, y], dim=1)

'''
class PredicateModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.activation = nn.Sigmoid()

    def forward(self, logits, class_idx):
        probs = self.activation(logits)  # [batch, num_classes]

        # Make sure class_idx is the same shape as batch
        if class_idx.dim() == 0:
            class_idx = class_idx.repeat(logits.shape[0])

        # Use torch.arange to select batch elements along dim 0
        selected = probs[torch.arange(probs.size(0)), class_idx.long()]

        #print(selected.shape)
        return selected  # [batch]




'''
        if class_idx.dim() == 0:
            class_idx = class_idx.repeat(logits.shape[0])

        class_idx = class_idx.long().unsqueeze(1)

        print("shapes")
        print(probs.shape)
        print(class_idx.shape)

        selected = torch.gather(probs, 1, class_idx)

        return selected.squeeze(1)      # [batch]
'''


        #return probs[:, class_idx].squeeze()         # select class

#num_classes = len(CLASSES)
#model = CNN(in_channels=3, num_classes=num_classes)


#x = torch.randn(1, 3, 224, 224)
#output = model(x)

#print(output.shape)


# Predicates
#final classes = 4
logits_model = CNN(in_channels=3, num_classes=final_classes)


# automatically use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# move model to device
logits_model = logits_model.to(device)
#missCounter = 0
classification_loss_fn = nn.BCEWithLogitsLoss()

p = ltn.Predicate(PredicateModel())


# Constants
#final classes
class_cafe = ltn.Constant(torch.tensor(CLASSES.index("Cafe")), trainable=False)
class_hotel = ltn.Constant(torch.tensor(CLASSES.index("Hotel")), trainable=False)
class_store = ltn.Constant(torch.tensor(CLASSES.index("Store")), trainable=False)
class_misc_commercial = ltn.Constant(torch.tensor(CLASSES.index("MiscCommercial")), trainable=False)

#concepts
class_awning = ltn.Constant(torch.tensor(CLASSES.index("Awning")), trainable=False)
class_billboard = ltn.Constant(torch.tensor(CLASSES.index("Billboard")), trainable=False)
class_vending_machine = ltn.Constant(torch.tensor(CLASSES.index("VendingMachine")), trainable=False)
class_statue = ltn.Constant(torch.tensor(CLASSES.index("Statue")), trainable=False)
class_table = ltn.Constant(torch.tensor(CLASSES.index("Table")), trainable=False)
class_wall_sign = ltn.Constant(torch.tensor(CLASSES.index("WallSign")), trainable=False)



#Connectives
Not = ltn.Connective(ltn.fuzzy_ops.NotStandard())
And = ltn.Connective(ltn.fuzzy_ops.AndProd())
Or = ltn.Connective(ltn.fuzzy_ops.OrProbSum())
Implies = ltn.Connective(ltn.fuzzy_ops.ImpliesReichenbach())
Forall = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")
Exists = ltn.Quantifier(ltn.fuzzy_ops.AggregPMean(p=2), quantifier="e")
Equiv = ltn.Connective(
    ltn.fuzzy_ops.Equiv(
        ltn.fuzzy_ops.AndProd(),
        ltn.fuzzy_ops.ImpliesReichenbach()
    )
)

formula_aggregator = ltn.fuzzy_ops.SatAgg(
    ltn.fuzzy_ops.AggregPMeanError(p=2)
)

#print("before axioms")

#missCounter = {"count": 0}

#axioms
def axioms(logits, *args):

    #logits = logits_model(features_param)

    x = ltn.Variable("x", logits)
    x_hotel = ltn.Variable("x_hotel", logits[args[CLASSES.index("Hotel")] == 1, :])
    x_not_hotel = ltn.Variable(
        "x_not_hotel", logits[args[CLASSES.index("Hotel")] == 0, :]
    )
    x_cafe = ltn.Variable("x_cafe", logits[args[CLASSES.index("Cafe")] == 1, :])
    x_not_cafe = ltn.Variable(
        "x_not_cafe", logits[args[CLASSES.index("Cafe")] == 0, :]
    )
    x_store = ltn.Variable("x_store", logits[args[CLASSES.index("Store")] == 1, :])
    x_not_store = ltn.Variable(
        "x_not_store", logits[args[CLASSES.index("Store")] == 0, :]
    )
    x_misc_commercial = ltn.Variable(
        "x_misc_commercial", logits[args[CLASSES.index("MiscCommercial")] == 1, :]
    )
    x_not_misc_commercial = ltn.Variable(
        "x_not_misc_commercial",
        logits[args[CLASSES.index("MiscCommercial")] == 0, :],
    )

    def safe_forall(var, formula, default_value=0.5):
    # Check if variable has zero elements
        
        if var.value.shape[0] == 0:  # no elements in the batch
            #print("\n\n\n\n\n\n\n\n miss \n\n\n\n\n\n")
            return ltn.Constant(torch.tensor(default_value, device=logits.device))
        else:
            return Forall(var, formula)



    axioms = [
        # Grounded Predicates

        safe_forall(x_hotel, p(x_hotel, class_hotel)),
        safe_forall(x_not_hotel, Not(p(x_not_hotel, class_hotel))),
        safe_forall(x_cafe, p(x_cafe, class_cafe)),
        safe_forall(x_not_cafe, Not(p(x_not_cafe, class_cafe))),
        safe_forall(x_store, p(x_store, class_store)),
        safe_forall(x_not_store, Not(p(x_not_store, class_store))),
        safe_forall(x_misc_commercial, p(x_misc_commercial, class_misc_commercial)),
        safe_forall(
            x_not_misc_commercial,
            Not(p(x_not_misc_commercial, class_misc_commercial)),
        ),
        # Axioms
        # Cafe = Statue or Vending Machine
        safe_forall(
            x,
            Equiv(
                p(x, class_cafe),
                Or(p(x, class_statue), p(x, class_vending_machine)),
            ),
        ),
        # Hotel = Wall Sign
        safe_forall(
            x,
            Equiv(p(x, class_hotel), p(x, class_wall_sign)),
        ),
        # MiscCommercial = Awning and Table
        safe_forall(
            x,
            Equiv(
                p(x, class_misc_commercial),
                And(p(x, class_awning), p(x, class_table)),
            ),
        ),
        # Store = Billboard
        safe_forall(
            x,
            Equiv(p(x, class_store), p(x, class_billboard)),
        ),
    ]
    #print("values of axioms")
    #for a in axioms:
      #print(a.value)

    sat_level = formula_aggregator(*axioms)

    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list

'''
for batch_elements in train_loader:
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(features, *labels_list))
    break
'''

""" TRAINING FUNCTIONS """

class Counter:
    def __init__(self):
        self.reset_state()

    def reset_state(self):
        self.value = 0.0

    def update(self, value):
        self.value = value

    def result(self):
        return self.value

class MeanMetric:
    def __init__(self):
        self.reset_state()

    def update(self, value, n=1):
        self.total += value * n
        self.count += n

    def compute(self):
        return self.total / self.count if self.count != 0 else 0.0


    def reset_state(self):
        self.total = 0.0
        self.count = 0

    def result(self):
        return self.compute()
    

class RunningAverage:
    def __init__(self):
        self.reset_state()

    def reset_state(self):
        self.total = 0
        self.count = 0

    def update(self, value):
        self.total += value
        self.count += 1

    
    def result(self):
        if self.count == 0:
            return 0
        return self.total / self.count


metrics_dict = {
    "train_sat_kb": MeanMetric(),
    "test_sat_kb": MeanMetric(),
    "train_loss": RunningAverage(),
    "test_loss": RunningAverage(),
    "alpha": Counter()
}

class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": MeanMetric()
    for class_name in CLASSES
}

metrics_dict.update(class_train_accuracy_metric)

#optimizer = torch.optim.Adam(logits_model.parameters(), lr=LEARNING_RATE)


def train_step(features_param, *labels_list, optimizer, alpha=0.5):

    logits_model.train()
    optimizer.zero_grad()

    logits = logits_model(features_param)

    # first 4 logits correspond to classes
    class_logits = logits[:, :4]

    # only first 4 labels
    class_labels = labels_list[:4]
    labels_tensor = torch.stack(class_labels, dim=1).float()

    classification_loss = classification_loss_fn(class_logits, labels_tensor)

    sat = axioms(logits, *labels_list)

    ltn_loss = 1.0 - sat

    # ----- Combined loss -----
    loss = alpha * classification_loss + (1 - alpha) * ltn_loss

    loss.backward()
    optimizer.step()

    print(f"Sat Function Result: {sat.item()}, Class_loss: {classification_loss}, ltn-loss: {ltn_loss}, loss:{loss} ")

    # Update metric
    metrics_dict["train_sat_kb"].update(sat.item())
    metrics_dict["train_loss"].update(classification_loss.item())

    #predictions = logits

    calculate_accuracy_for_each_class(logits, labels_list)

@torch.no_grad()
def test_step(features_param, *labels_list, alpha=0.5):
    logits_model.eval()

    logits = logits_model(features_param)
        
    # first 4 logits correspond to classes
    class_logits = logits[:, :4]

    # only first 4 labels
    class_labels = labels_list[:4]
    labels_tensor = torch.stack(class_labels, dim=1).float()

    classification_loss = classification_loss_fn(class_logits, labels_tensor)

    sat_kb = axioms(logits, *labels_list)

    ltn_loss = 1.0 - sat_kb

    # ----- Combined loss -----
    loss = sat_kb * classification_loss + (1 - alpha) * ltn_loss


    metrics_dict["test_sat_kb"].update(sat_kb.item())
    metrics_dict["test_loss"].update(loss.item())


""" AUXILIARY FUNCTIONS """

def calculate_accuracy_for_each_class(predictions, labels_param):
    probs = torch.sigmoid(predictions)

    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i].float()

        y_pred = probs[:, i]
        y_pred = (y_pred > 0.5).float()

        accuracy = (y_pred == class_labels).float().mean()

        metrics_dict[f"{class_name.lower()}_train_accuracy"].update(
            accuracy.item()
        )


def write_results_description_in_txt(txt_file, model):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")

    if model is not None:
        txt_file.write("\nModel Architecture:\n")
        txt_file.write(str(model))
        txt_file.write("\n\n")


#final_epochs: epochs with final alpha
def new_calculate_alpha(epoch= 0, warmup_epochs = 10, final_alpha = 0.25, final_epochs=30):
    alpha = 1.0

    if epoch < warmup_epochs:
        alpha = 1.0
    elif epoch < EPOCHS - final_epochs:
        progress = (epoch - warmup_epochs) / (EPOCHS - warmup_epochs-final_epochs)
        alpha = 1.0 - progress * (1.0 - final_alpha)
    else:
        alpha = final_alpha

    return alpha


""" SCRIPT """
LIMIT = 1500
EPOCHS = 80 #era 150
#EARLY_STOPPING_DELTA = 0.005
#PATIENCE = 20
MODEL_SAVING_INTERVAL = 10

def train_loop(warmup_epochs = 10, final_alpha = 0.25, final_epochs=30, patience=20, early_stopping_delta=0.001, learning_rate=0.001, train_model_number=0):

    #folder
    run_name = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    OUTPUT_DIR = Path(f"outputs{train_model_number}_{run_name}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    csv_path = OUTPUT_DIR / "metrics.csv"
    txt_path = OUTPUT_DIR / "stats.txt"

    metrics_dict.update(class_train_accuracy_metric)

    optimizer = torch.optim.Adam(logits_model.parameters(), lr=learning_rate)

    best_sat = 0.0
    epochs_no_improve = 0

    # scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
    template = "Epoch {}"
    for metrics_label in metrics_dict.keys():
        template += ", %s: {:.5f}" % metrics_label
    if csv_path is not None:
        txt_file = open(txt_path, "w+")
        write_results_description_in_txt(txt_file, logits_model)

        csv_file = open(csv_path, "w+")
        headers = ",".join(["Epoch", "Alpha"] + list(metrics_dict.keys())) #sem o alpha
        csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 2)]) #era +1
        csv_file.write(headers + "\n")

    for epoch in range(LIMIT):
        print(f"starting epoch {epoch}")
        #reset metrics
        for metrics in metrics_dict.values():
            metrics.reset_state()

        #alpha = new_calculate_alpha(epoch, warmup_epochs, final_alpha, final_epochs)
        alpha = 0.5
        metrics_dict["alpha"].update(alpha)

        print(f"alpha: {alpha}")

        for batch_elements in train_loader:
            features_param, labels_list, ids = __get_elements_from_batch(batch_elements)

            #print("labels list")
            #print(labels_list.shape)

            # Move to device if needed
            features_param = features_param.to(device)
            labels_list = [lbl.to(device) for lbl in labels_list]

            #print("train step")
            train_step(features_param, *labels_list, optimizer=optimizer, alpha=alpha)


        for batch_elements_test in test_loader:
            features_param, labels_list = __get_elements_from_batch(batch_elements_test)

            # Move to device if needed
            features_param = features_param.to(device)
            labels_list = [lbl.to(device) for lbl in labels_list]

            test_step(features_param, *labels_list, alpha=alpha)

        metrics_results = [metrics.result() for metrics in metrics_dict.values()]
        print(template.format(epoch, *metrics_results))
        if csv_path is not None:
            #csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
            csv_file.write(csv_template.format(epoch, alpha, *metrics_results) + "\n")
            csv_file.flush()

        #early stopping condition
        current_sat = metrics_dict["test_sat_kb"].result()
        new_sat = best_sat + early_stopping_delta

        print(f"best_sat: {best_sat}, current_sat: {current_sat}")

        if current_sat > new_sat:
            best_sat = current_sat
            epochs_no_improve = 0
            print("\n\n\n CHANGED BEST SAT \n\n\n")
        elif epoch > 0:
            #print("\n\n\n CHANGED BEST SAT \n\n\n")
            epochs_no_improve += 1


        if epochs_no_improve >= patience:
            print("EARLY STOPPING TRIGGERED")
            break

        #model saving
        if epoch != 0 and epoch % MODEL_SAVING_INTERVAL == 0:
            print(f"SAVING MODEL AT EPOCH {epoch}")
            torch.save(logits_model.state_dict(), OUTPUT_DIR / f"model_epoch_{epoch}.pth")
        




    print("training finished, starting to save the model")
    torch.save(logits_model.state_dict(), OUTPUT_DIR / f"model_epoch_{epoch}.pth")

    if csv_path is not None:
        csv_file.close()
        txt_file.close()


#train loop

train_loop(warmup_epochs = 10, final_alpha = 0.25, final_epochs=30, patience=10, early_stopping_delta=0.005, learning_rate=LEARNING_RATE, train_model_number=0)

starting epoch 0
alpha: 0.5
Sat Function Result: 0.517501711845398, Class_loss: 0.7033408880233765, ltn-loss: 0.48249828815460205, loss:0.5929195880889893 
Sat Function Result: 0.5178373456001282, Class_loss: 0.6728714108467102, ltn-loss: 0.4821626543998718, loss:0.577517032623291 
Sat Function Result: 0.5077962875366211, Class_loss: 0.644486665725708, ltn-loss: 0.4922037124633789, loss:0.5683451890945435 
Sat Function Result: 0.4906032681465149, Class_loss: 0.5833842754364014, ltn-loss: 0.5093967318534851, loss:0.5463905334472656 
Sat Function Result: 0.46357667446136475, Class_loss: 0.6121370792388916, ltn-loss: 0.5364233255386353, loss:0.5742802023887634 
Sat Function Result: 0.47863221168518066, Class_loss: 0.5088945627212524, ltn-loss: 0.5213677883148193, loss:0.5151311755180359 
Sat Function Result: 0.49190855026245117, Class_loss: 0.4908389449119568, ltn-loss: 0.5080914497375488, loss:0.4994651973247528 
Sat Function Result: 0.46502751111984253, Class_loss: 0.5630044341087341, l

KeyboardInterrupt: 

script completo

In [ ]:

data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(DATA_ROOT, "all.csv")
images = Path(DATA_ROOT, "c1")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]
CLASSES = [
    "Cafe",
    "Hotel",
    "Store",
    "MiscCommercial",
    "Awning",
    "Billboard",
    "VendingMachine",
    "Statue",
    "Table",
    "WallSign",
]

class BuildingsDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = self.img_dir / row["id"]
        image = Image.open(img_path).convert("RGB")



        #target = torch.tensor(row[CLASSES].values, dtype=torch.float32)
        target = torch.tensor(row[CLASSES].to_numpy(dtype=float), dtype=torch.float32)

        #print(row[CLASSES].to_numpy().dtype)


        if self.transform:
            image = self.transform(image)

        return image, target


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


TRAIN_TEST_LIMIT = int(0.8 * len(df))

train_df = df[:TRAIN_TEST_LIMIT]
test_df = df[TRAIN_TEST_LIMIT:]

df[CLASSES] = df[CLASSES].apply(pd.to_numeric)


train_dataset = BuildingsDataset(train_df, images, transform=transform)
test_dataset = BuildingsDataset(test_df, images, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

#print(train_df["Cafe"])

"""CNN MODEL"""

class CNN(nn.Module):
   def __init__(self, in_channels, num_classes, name="cnn_model"):

       """
       Building blocks of convolutional neural network.

       Parameters:
           * in_channels: Number of channels in the input image (for grayscale images, 1)
           * num_classes: Number of classes to predict. In our problem, 10 (i.e digits from  0 to 9).
       """
       super(CNN, self).__init__()

       #layers
       self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3)
       self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3)
       self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
       #self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3)

       self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
       self.flatten = nn.Flatten()
       self.dropout = nn.Dropout(0.5)

        #3 conv 43624, 4 conv 18432
       self.dense1 = nn.Linear(43264, 128)#com 3 conv, 43264
       self.dense2 = nn.Linear(128, 64)

       self.concept_layer = nn.Linear(64, 32)
       self.output_layer = nn.Linear(32, num_classes)


   def forward(self, inputs):
       """
       Define the forward pass of the neural network.

       Parameters:
           x: Input tensor.

       Returns:
           torch.Tensor
               The output tensor after passing through the network.
       """
       if isinstance(inputs, (list, tuple)):
          inputs = inputs[0]


       x = F.relu(self.conv1(inputs))
       x = self.maxpool(x)
       x = F.relu(self.conv2(x))
       x = self.maxpool(x)
       x = F.relu(self.conv3(x))
       x = self.maxpool(x)

       x = self.flatten(x)
       x = self.dropout(x)

       x = F.relu(self.dense1(x))
       x = self.dropout(x)
       x = F.relu(self.dense2(x))

       y = F.relu(self.concept_layer(x))
       x = self.output_layer(y)

       result = torch.cat([x, y], dim=1)
       return result

      
'''
class CNN(nn.Module):
    def __init__(self, in_channels, num_classes, name="cnn_model"):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, 3)
        self.conv2 = nn.Conv2d(16, 32, 3)
        self.conv3 = nn.Conv2d(32, 64, 3)

        self.pool = nn.MaxPool2d(2)

        self.gap = nn.AdaptiveAvgPool2d((1,1))

        self.fc1 = nn.Linear(64, 128)
        self.fc2 = nn.Linear(128, 64)

        self.concept_layer = nn.Linear(64, 32)
        self.output_layer = nn.Linear(32, num_classes)

    def forward(self, x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = self.gap(x)
        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        y = F.relu(self.concept_layer(x))
        x = self.output_layer(y)

        return torch.cat([x, y], dim=1)

'''
class PredicateModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.activation = nn.Sigmoid()

    def forward(self, logits, class_idx):
        probs = self.activation(logits)  # [batch, num_classes]

        # Make sure class_idx is the same shape as batch
        if class_idx.dim() == 0:
            class_idx = class_idx.repeat(logits.shape[0])

        # Use torch.arange to select batch elements along dim 0
        selected = probs[torch.arange(probs.size(0)), class_idx.long()]

        #print(selected.shape)
        return selected  # [batch]




'''
        if class_idx.dim() == 0:
            class_idx = class_idx.repeat(logits.shape[0])

        class_idx = class_idx.long().unsqueeze(1)

        print("shapes")
        print(probs.shape)
        print(class_idx.shape)

        selected = torch.gather(probs, 1, class_idx)

        return selected.squeeze(1)      # [batch]
'''


        #return probs[:, class_idx].squeeze()         # select class

#num_classes = len(CLASSES)
#model = CNN(in_channels=3, num_classes=num_classes)


#x = torch.randn(1, 3, 224, 224)
#output = model(x)

#print(output.shape)


# Predicates
logits_model = CNN(in_channels=3, num_classes=4)


# automatically use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# move model to device
logits_model = logits_model.to(device)
#missCounter = 0

p = ltn.Predicate(PredicateModel())


# Constants
class_cafe = ltn.Constant(torch.tensor(CLASSES.index("Cafe")), trainable=False)
class_hotel = ltn.Constant(torch.tensor(CLASSES.index("Hotel")), trainable=False)
class_store = ltn.Constant(torch.tensor(CLASSES.index("Store")), trainable=False)
class_misc_commercial = ltn.Constant(torch.tensor(CLASSES.index("MiscCommercial")), trainable=False)
class_awning = ltn.Constant(torch.tensor(CLASSES.index("Awning")), trainable=False)
class_billboard = ltn.Constant(torch.tensor(CLASSES.index("Billboard")), trainable=False)
class_vending_machine = ltn.Constant(torch.tensor(CLASSES.index("VendingMachine")), trainable=False)
class_statue = ltn.Constant(torch.tensor(CLASSES.index("Statue")), trainable=False)
class_table = ltn.Constant(torch.tensor(CLASSES.index("Table")), trainable=False)
class_wall_sign = ltn.Constant(torch.tensor(CLASSES.index("WallSign")), trainable=False)



#Connectives
Not = ltn.Connective(ltn.fuzzy_ops.NotStandard())
And = ltn.Connective(ltn.fuzzy_ops.AndProd())
Or = ltn.Connective(ltn.fuzzy_ops.OrProbSum())
Implies = ltn.Connective(ltn.fuzzy_ops.ImpliesReichenbach())
Forall = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")
Exists = ltn.Quantifier(ltn.fuzzy_ops.AggregPMean(p=2), quantifier="e")
Equiv = ltn.Connective(
    ltn.fuzzy_ops.Equiv(
        ltn.fuzzy_ops.AndProd(),
        ltn.fuzzy_ops.ImpliesReichenbach()
    )
)

formula_aggregator = ltn.fuzzy_ops.SatAgg(
    ltn.fuzzy_ops.AggregPMeanError(p=2)
)

#print("before axioms")

#missCounter = {"count": 0}

#axioms
def axioms(logits, *args):

    #logits = logits_model(features_param)

    x = ltn.Variable("x", logits)
    x_hotel = ltn.Variable("x_hotel", logits[args[CLASSES.index("Hotel")] == 1, :])
    x_not_hotel = ltn.Variable(
        "x_not_hotel", logits[args[CLASSES.index("Hotel")] == 0, :]
    )
    x_cafe = ltn.Variable("x_cafe", logits[args[CLASSES.index("Cafe")] == 1, :])
    x_not_cafe = ltn.Variable(
        "x_not_cafe", logits[args[CLASSES.index("Cafe")] == 0, :]
    )
    x_store = ltn.Variable("x_store", logits[args[CLASSES.index("Store")] == 1, :])
    x_not_store = ltn.Variable(
        "x_not_store", logits[args[CLASSES.index("Store")] == 0, :]
    )
    x_misc_commercial = ltn.Variable(
        "x_misc_commercial", logits[args[CLASSES.index("MiscCommercial")] == 1, :]
    )
    x_not_misc_commercial = ltn.Variable(
        "x_not_misc_commercial",
        logits[args[CLASSES.index("MiscCommercial")] == 0, :],
    )

    def safe_forall(var, formula, default_value=0.5):
    # Check if variable has zero elements
        
        if var.value.shape[0] == 0:  # no elements in the batch
            # Return a neutral truth value (0.5)
            print("\n\n\n\n\n\n\n\n miss \n\n\n\n\n\n")

            #missCounter += 1
            return ltn.Constant(torch.tensor(default_value, device=logits.device))
        else:
            return Forall(var, formula)



    axioms = [
        # Grounded Predicates

        safe_forall(x_hotel, p(x_hotel, class_hotel)),
        safe_forall(x_not_hotel, Not(p(x_not_hotel, class_hotel))),
        safe_forall(x_cafe, p(x_cafe, class_cafe)),
        safe_forall(x_not_cafe, Not(p(x_not_cafe, class_cafe))),
        safe_forall(x_store, p(x_store, class_store)),
        safe_forall(x_not_store, Not(p(x_not_store, class_store))),
        safe_forall(x_misc_commercial, p(x_misc_commercial, class_misc_commercial)),
        safe_forall(
            x_not_misc_commercial,
            Not(p(x_not_misc_commercial, class_misc_commercial)),
        ),
        # Axioms
        # Cafe = Statue or Vending Machine
        safe_forall(
            x,
            Equiv(
                p(x, class_cafe),
                Or(p(x, class_statue), p(x, class_vending_machine)),
            ),
        ),
        # Hotel = Wall Sign
        safe_forall(
            x,
            Equiv(p(x, class_hotel), p(x, class_wall_sign)),
        ),
        # MiscCommercial = Awning and Table
        safe_forall(
            x,
            Equiv(
                p(x, class_misc_commercial),
                And(p(x, class_awning), p(x, class_table)),
            ),
        ),
        # Store = Billboard
        safe_forall(
            x,
            Equiv(p(x, class_store), p(x, class_billboard)),
        ),
    ]
    #print("values of axioms")
    #for a in axioms:
      #print(a.value)

    sat_level = formula_aggregator(*axioms)

    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list

'''
for batch_elements in train_loader:
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(features, *labels_list))
    break
'''

""" TRAINING FUNCTIONS """

class MeanMetric:
    def __init__(self):
        self.reset_state()

    def update(self, value, n=1):
        self.total += value * n
        self.count += n

    def compute(self):
        return self.total / self.count if self.count != 0 else 0.0

    def reset_state(self):
        self.total = 0.0
        self.count = 0

    def result(self):
        return self.compute()

metrics_dict = {
    "train_sat_kb": MeanMetric(),
    "test_sat_kb": MeanMetric(),
}

class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": MeanMetric()
    for class_name in CLASSES
}

metrics_dict.update(class_train_accuracy_metric)

optimizer = torch.optim.Adam(logits_model.parameters(), lr=LEARNING_RATE)


def train_step(features_param, *labels_list):

    logits_model.train()
    optimizer.zero_grad()

    logits = logits_model(features_param)

    #print("axioms")
    # Forward through axioms (LTN)
    sat = axioms(logits, *labels_list)

    #print("sat")
    #print(sat)
    loss = 1.0 - sat
    #print("loss")
    #print(loss)
    #print(sat.value)

    # Backprop
    #print ("before backward")
    loss.backward()
    #print("after backward")
    optimizer.step()
    #print("after optimizer")

    print("Sat Function Result:", sat.item())

    # Update metric
    metrics_dict["train_sat_kb"].update(sat.item())

    # Predictions
    #predictions = logits_model(features_param)
    predictions = logits

    calculate_accuracy_for_each_class(predictions, labels_list)

@torch.no_grad()
def test_step(features_param, *labels_list):
    logits_model.eval()

    logits = logits_model(features_param)

    sat_kb = axioms(logits, *labels_list)

    metrics_dict["test_sat_kb"].update(sat_kb.item())


""" AUXILIARY FUNCTIONS """

def calculate_accuracy_for_each_class(predictions, labels_param):
    probs = torch.sigmoid(predictions)

    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i].float()

        y_pred = probs[:, i]
        y_pred = (y_pred > 0.5).float()

        accuracy = (y_pred == class_labels).float().mean()

        metrics_dict[f"{class_name.lower()}_train_accuracy"].update(
            accuracy.item()
        )


def write_results_description_in_txt(txt_file, model):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")

    if model is not None:
        txt_file.write("\nModel Architecture:\n")
        txt_file.write(str(model))
        txt_file.write("\n\n")




""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file, logits_model)

    csv_file = open(csv_path, "w+")
    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    print(f"starting epoch {epoch}")
    #reset metrics
    for metrics in metrics_dict.values():
        metrics.reset_state()

    print("after metrics")

    for batch_elements in train_loader:
        features_param, labels_list = __get_elements_from_batch(batch_elements)

        # Move to device if needed
        features_param = features_param.to(device)
        labels_list = [lbl.to(device) for lbl in labels_list]

        #print("train step")
        train_step(features_param, *labels_list)


    for batch_elements_test in test_loader:
        features_param, labels_list = __get_elements_from_batch(batch_elements_test)

        # Move to device if needed
        features_param = features_param.to(device)
        labels_list = [lbl.to(device) for lbl in labels_list]

        test_step(features_param, *labels_list)

    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()

print("training finished, starting to save the model")
torch.save(logits_model.state_dict(), f"model_epoch_{epoch}.pth")

if csv_path is not None:
    csv_file.close()
    txt_file.close()

In [21]:
torch.save(logits_model.state_dict(), f"model_epoch_{epoch}.pth")

com resnet18

In [18]:

data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(DATA_ROOT, "all.csv")
images = Path(DATA_ROOT, "c1")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]
CLASSES = [
    "Cafe",
    "Hotel",
    "Store",
    "MiscCommercial",
    "Awning",
    "Billboard",
    "VendingMachine",
    "Statue",
    "Table",
    "WallSign",
]

class BuildingsDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = self.img_dir / row["id"]
        image = Image.open(img_path).convert("RGB")



        #target = torch.tensor(row[CLASSES].values, dtype=torch.float32)
        target = torch.tensor(row[CLASSES].to_numpy(dtype=float), dtype=torch.float32)

        #print(row[CLASSES].to_numpy().dtype)


        if self.transform:
            image = self.transform(image)

        return image, target


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


TRAIN_TEST_LIMIT = int(0.8 * len(df))

train_df = df[:TRAIN_TEST_LIMIT]
test_df = df[TRAIN_TEST_LIMIT:]

df[CLASSES] = df[CLASSES].apply(pd.to_numeric)


train_dataset = BuildingsDataset(train_df, images, transform=transform)
test_dataset = BuildingsDataset(test_df, images, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

#print(train_df["Cafe"])

"""CNN MODEL"""
class CNN(nn.Module):
   def __init__(self, in_channels, num_classes, name="cnn_model"):

       """
       Building blocks of convolutional neural network.

       Parameters:
           * in_channels: Number of channels in the input image (for grayscale images, 1)
           * num_classes: Number of classes to predict. In our problem, 10 (i.e digits from  0 to 9).
       """
       super(CNN, self).__init__()

       #layers
       self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3)
       self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3)
       self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)

       self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
       self.flatten = nn.Flatten()
       self.dropout = nn.Dropout(0.5)

       self.dense1 = nn.Linear(43264, 128)
       self.dense2 = nn.Linear(128, 64)
       self.concept_layer = nn.Linear(64, 32)
       self.output_layer = nn.Linear(32, num_classes)


   def forward(self, inputs):
       """
       Define the forward pass of the neural network.

       Parameters:
           x: Input tensor.

       Returns:
           torch.Tensor
               The output tensor after passing through the network.
       """
       if isinstance(inputs, (list, tuple)):
          inputs = inputs[0]


       x = F.relu(self.conv1(inputs))
       x = self.maxpool(x)
       x = F.relu(self.conv2(x))
       x = self.maxpool(x)
       x = F.relu(self.conv3(x))
       x = self.maxpool(x)

       x = self.flatten(x)
       x = self.dropout(x)

       x = F.relu(self.dense1(x))
       x = self.dropout(x)
       x = F.relu(self.dense2(x))

       y = F.relu(self.concept_layer(x))
       x = self.output_layer(y)

       result = torch.cat([x, y], dim=1)
       return result


class PredicateModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.activation = nn.Sigmoid()

    def forward(self, logits, class_idx):
        probs = self.activation(logits)  # [batch, num_classes]

        # Make sure class_idx is the same shape as batch
        if class_idx.dim() == 0:
            class_idx = class_idx.repeat(logits.shape[0])

        # Use torch.arange to select batch elements along dim 0
        selected = probs[torch.arange(probs.size(0)), class_idx.long()]

        #print(selected.shape)
        return selected  # [batch]
    

class ResNetLTN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.backbone = models.resnet18(weights="IMAGENET1K_V1")

        # remove default classifier
        self.backbone.fc = nn.Identity()

        self.concept_layer = nn.Linear(512, 64)
        self.output_layer = nn.Linear(64, num_classes)

    def forward(self, x):

        features = self.backbone(x)   # 512 feature vector

        concepts = torch.relu(self.concept_layer(features))
        logits = self.output_layer(concepts)

        result = torch.cat([logits, concepts], dim=1)

        return result




'''
        if class_idx.dim() == 0:
            class_idx = class_idx.repeat(logits.shape[0])

        class_idx = class_idx.long().unsqueeze(1)

        print("shapes")
        print(probs.shape)
        print(class_idx.shape)

        selected = torch.gather(probs, 1, class_idx)

        return selected.squeeze(1)      # [batch]
'''


        #return probs[:, class_idx].squeeze()         # select class

#num_classes = len(CLASSES)
#model = CNN(in_channels=3, num_classes=num_classes)


#x = torch.randn(1, 3, 224, 224)
#output = model(x)

#print(output.shape)
resnet = models.resnet18(weights="IMAGENET1K_V1")

# Predicates
logits_model = ResNetLTN(num_classes=4)


# automatically use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# move model to device
logits_model = logits_model.to(device)
#missCounter = 0

p = ltn.Predicate(PredicateModel())


# Constants
class_cafe = ltn.Constant(torch.tensor(CLASSES.index("Cafe")), trainable=False)
class_hotel = ltn.Constant(torch.tensor(CLASSES.index("Hotel")), trainable=False)
class_store = ltn.Constant(torch.tensor(CLASSES.index("Store")), trainable=False)
class_misc_commercial = ltn.Constant(torch.tensor(CLASSES.index("MiscCommercial")), trainable=False)
class_awning = ltn.Constant(torch.tensor(CLASSES.index("Awning")), trainable=False)
class_billboard = ltn.Constant(torch.tensor(CLASSES.index("Billboard")), trainable=False)
class_vending_machine = ltn.Constant(torch.tensor(CLASSES.index("VendingMachine")), trainable=False)
class_statue = ltn.Constant(torch.tensor(CLASSES.index("Statue")), trainable=False)
class_table = ltn.Constant(torch.tensor(CLASSES.index("Table")), trainable=False)
class_wall_sign = ltn.Constant(torch.tensor(CLASSES.index("WallSign")), trainable=False)



#Connectives
Not = ltn.Connective(ltn.fuzzy_ops.NotStandard())
And = ltn.Connective(ltn.fuzzy_ops.AndProd())
Or = ltn.Connective(ltn.fuzzy_ops.OrProbSum())
Implies = ltn.Connective(ltn.fuzzy_ops.ImpliesReichenbach())
Forall = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")
Exists = ltn.Quantifier(ltn.fuzzy_ops.AggregPMean(p=2), quantifier="e")
Equiv = ltn.Connective(
    ltn.fuzzy_ops.Equiv(
        ltn.fuzzy_ops.AndProd(),
        ltn.fuzzy_ops.ImpliesReichenbach()
    )
)

formula_aggregator = ltn.fuzzy_ops.SatAgg(
    ltn.fuzzy_ops.AggregPMeanError(p=2)
)

#print("before axioms")

#missCounter = {"count": 0}

#axioms
def axioms(logits, *args):

    #logits = logits_model(features_param)

    x = ltn.Variable("x", logits)
    x_hotel = ltn.Variable("x_hotel", logits[args[CLASSES.index("Hotel")] == 1, :])
    x_not_hotel = ltn.Variable(
        "x_not_hotel", logits[args[CLASSES.index("Hotel")] == 0, :]
    )
    x_cafe = ltn.Variable("x_cafe", logits[args[CLASSES.index("Cafe")] == 1, :])
    x_not_cafe = ltn.Variable(
        "x_not_cafe", logits[args[CLASSES.index("Cafe")] == 0, :]
    )
    x_store = ltn.Variable("x_store", logits[args[CLASSES.index("Store")] == 1, :])
    x_not_store = ltn.Variable(
        "x_not_store", logits[args[CLASSES.index("Store")] == 0, :]
    )
    x_misc_commercial = ltn.Variable(
        "x_misc_commercial", logits[args[CLASSES.index("MiscCommercial")] == 1, :]
    )
    x_not_misc_commercial = ltn.Variable(
        "x_not_misc_commercial",
        logits[args[CLASSES.index("MiscCommercial")] == 0, :],
    )

    def safe_forall(var, formula, default_value=0.5):
    # Check if variable has zero elements
        
        if var.value.shape[0] == 0:  # no elements in the batch
            # Return a neutral truth value (0.5)
            print("\n\n\n\n\n\n\n\n miss \n\n\n\n\n\n")

            #missCounter += 1
            return torch.tensor(default_value, device="cuda" if torch.cuda.is_available() else "cpu", requires_grad=True)
        else:
            return Forall(var, formula)



    axioms = [
        # Grounded Predicates

        safe_forall(x_hotel, p(x_hotel, class_hotel)),
        safe_forall(x_not_hotel, Not(p(x_not_hotel, class_hotel))),
        safe_forall(x_cafe, p(x_cafe, class_cafe)),
        safe_forall(x_not_cafe, Not(p(x_not_cafe, class_cafe))),
        safe_forall(x_store, p(x_store, class_store)),
        safe_forall(x_not_store, Not(p(x_not_store, class_store))),
        safe_forall(x_misc_commercial, p(x_misc_commercial, class_misc_commercial)),
        safe_forall(
            x_not_misc_commercial,
            Not(p(x_not_misc_commercial, class_misc_commercial)),
        ),
        # Axioms
        # Cafe = Statue or Vending Machine
        safe_forall(
            x,
            Equiv(
                p(x, class_cafe),
                Or(p(x, class_statue), p(x, class_vending_machine)),
            ),
        ),
        # Hotel = Wall Sign
        safe_forall(
            x,
            Equiv(p(x, class_hotel), p(x, class_wall_sign)),
        ),
        # MiscCommercial = Awning and Table
        safe_forall(
            x,
            Equiv(
                p(x, class_misc_commercial),
                And(p(x, class_awning), p(x, class_table)),
            ),
        ),
        # Store = Billboard
        safe_forall(
            x,
            Equiv(p(x, class_store), p(x, class_billboard)),
        ),
    ]
    #print("values of axioms")
    #for a in axioms:
      #print(a.value)

    sat_level = formula_aggregator(*axioms)

    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list

'''
for batch_elements in train_loader:
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(features, *labels_list))
    break
'''

""" TRAINING FUNCTIONS """

class MeanMetric:
    def __init__(self):
        self.reset_state()

    def update(self, value, n=1):
        self.total += value * n
        self.count += n

    def compute(self):
        return self.total / self.count if self.count != 0 else 0.0

    def reset_state(self):
        self.total = 0.0
        self.count = 0

    def result(self):
        return self.compute()

metrics_dict = {
    "train_sat_kb": MeanMetric(),
    "test_sat_kb": MeanMetric(),
}

class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": MeanMetric()
    for class_name in CLASSES
}

metrics_dict.update(class_train_accuracy_metric)

optimizer = torch.optim.Adam(logits_model.parameters(), lr=LEARNING_RATE)


def train_step(features_param, *labels_list):

    logits_model.train()
    optimizer.zero_grad()

    logits = logits_model(features_param)

    # Forward through axioms (LTN)
    sat = axioms(logits, *labels_list)

    loss = 1.0 - sat

    # Backprop
    loss.backward()
    optimizer.step()

    print("Sat Function Result:", sat.item())

    # Update metric
    metrics_dict["train_sat_kb"].update(sat.item())

    # Predictions
    predictions = logits_model(features_param)

    calculate_accuracy_for_each_class(predictions, labels_list)

@torch.no_grad()
def test_step(features_param, *labels_list):
    logits_model.eval()

    logits = logits_model(features_param)

    sat_kb = axioms(logits, *labels_list)

    metrics_dict["test_sat_kb"].update(sat_kb.item())


""" AUXILIARY FUNCTIONS """

def calculate_accuracy_for_each_class(predictions, labels_param):
    probs = torch.sigmoid(predictions)

    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i].float()

        y_pred = probs[:, i]
        y_pred = (y_pred > 0.5).float()

        accuracy = (y_pred == class_labels).float().mean()

        metrics_dict[f"{class_name.lower()}_train_accuracy"].update(
            accuracy.item()
        )


def write_results_description_in_txt(txt_file):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")




""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file)

    csv_file = open(csv_path, "w+")
    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    print(f"starting epoch {epoch}")
    #reset metrics
    for metrics in metrics_dict.values():
        metrics.reset_state()

    for batch_elements in train_loader:
        features_param, labels_list = __get_elements_from_batch(batch_elements)

        # Move to device if needed
        features_param = features_param.to(device)
        labels_list = [lbl.to(device) for lbl in labels_list]

        train_step(features_param, *labels_list)


    for batch_elements_test in test_loader:
        features_param, labels_list = __get_elements_from_batch(batch_elements_test)

        # Move to device if needed
        features_param = features_param.to(device)
        labels_list = [lbl.to(device) for lbl in labels_list]

        test_step(features_param, *labels_list)

    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()


torch.save(logits_model.state_dict(), f"model_epoch_{epoch}.pth")

if csv_path is not None:
    csv_file.close()
    txt_file.close()

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/sofia/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


URLError: <urlopen error [Errno -3] Temporary failure in name resolution>

baseline

In [ ]:

data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(DATA_ROOT, "all.csv")
images = Path(DATA_ROOT, "c1")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]
CLASSES = [
    "Cafe",
    "Hotel",
    "Store",
    "MiscCommercial",
]

class BuildingsDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = self.img_dir / row["id"]
        image = Image.open(img_path).convert("RGB")



        #target = torch.tensor(row[CLASSES].values, dtype=torch.float32)
        target = torch.tensor(row[CLASSES].to_numpy(dtype=float), dtype=torch.float32)

        #print(row[CLASSES].to_numpy().dtype)


        if self.transform:
            image = self.transform(image)

        return image, target


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


TRAIN_TEST_LIMIT = int(0.8 * len(df))

train_df = df[:TRAIN_TEST_LIMIT]
test_df = df[TRAIN_TEST_LIMIT:]

df[CLASSES] = df[CLASSES].apply(pd.to_numeric)


train_dataset = BuildingsDataset(train_df, images, transform=transform)
test_dataset = BuildingsDataset(test_df, images, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

#print(train_df["Cafe"])

"""CNN MODEL"""
class CNN(nn.Module):
   def __init__(self, in_channels, num_classes, name="cnn_model"):

       """
       Building blocks of convolutional neural network.

       Parameters:
           * in_channels: Number of channels in the input image (for grayscale images, 1)
           * num_classes: Number of classes to predict. In our problem, 10 (i.e digits from  0 to 9).
       """
       super(CNN, self).__init__()

       #layers
       self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3)
       self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3)
       self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)

       self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
       self.flatten = nn.Flatten()
       self.dropout = nn.Dropout(0.5)

       self.dense1 = nn.Linear(43264, 128)
       self.dense2 = nn.Linear(128, 64)
       self.concept_layer = nn.Linear(64, 32)
       self.output_layer = nn.Linear(32, num_classes)


   def forward(self, inputs):
       """
       Define the forward pass of the neural network.

       Parameters:
           x: Input tensor.

       Returns:
           torch.Tensor
               The output tensor after passing through the network.
       """
       if isinstance(inputs, (list, tuple)):
          inputs = inputs[0]


       x = F.relu(self.conv1(inputs))
       x = self.maxpool(x)
       x = F.relu(self.conv2(x))
       x = self.maxpool(x)
       x = F.relu(self.conv3(x))
       x = self.maxpool(x)

       x = self.flatten(x)
       x = self.dropout(x)

       x = F.relu(self.dense1(x))
       x = self.dropout(x)
       x = F.relu(self.dense2(x))

       x = F.relu(self.concept_layer(x))
       x = self.output_layer(x)

       #result = torch.cat([x, y], dim=1)
       return x





'''
        if class_idx.dim() == 0:
            class_idx = class_idx.repeat(logits.shape[0])

        class_idx = class_idx.long().unsqueeze(1)

        print("shapes")
        print(probs.shape)
        print(class_idx.shape)

        selected = torch.gather(probs, 1, class_idx)

        return selected.squeeze(1)      # [batch]
'''


        #return probs[:, class_idx].squeeze()         # select class

#num_classes = len(CLASSES)
#model = CNN(in_channels=3, num_classes=num_classes)


#x = torch.randn(1, 3, 224, 224)
#output = model(x)

#print(output.shape)

criterion = torch.nn.BCEWithLogitsLoss()

# Predicates
logits_model = CNN(in_channels=3, num_classes=4)


# automatically use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# move model to device
logits_model = logits_model.to(device)


#print("before axioms")




'''
for batch_elements in train_loader:
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(features, *labels_list))
    break
'''

""" TRAINING FUNCTIONS """

class RunningAverage:
    def __init__(self):
        self.reset_state()

    def reset_state(self):
        self.total = 0
        self.count = 0

    def update(self, value):
        self.total += value
        self.count += 1

    
    def result(self):
        if self.count == 0:
            return 0
        return self.total / self.count

class MeanMetric:
    def __init__(self):
        self.reset_state()

    def update(self, value, n=1):
        self.total += value * n
        self.count += n

    def compute(self):
        return self.total / self.count if self.count != 0 else 0.0

    def reset_state(self):
        self.total = 0.0
        self.count = 0

    def result(self):
        return self.compute()

metrics_dict = {
    "train_loss": RunningAverage(),
    "test_loss": RunningAverage()
}

class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": MeanMetric()
    for class_name in CLASSES
}

metrics_dict.update(class_train_accuracy_metric)

optimizer = torch.optim.Adam(logits_model.parameters(), lr=LEARNING_RATE)


def train_step(features_param, labels):

    logits_model.train()
    optimizer.zero_grad()

    logits = logits_model(features_param)

    loss = criterion(logits, labels)

    print(f"current loss: {loss}")

    loss.backward()
    optimizer.step()

    metrics_dict["train_loss"].update(loss.item())

    calculate_accuracy_for_each_class(logits, labels)


@torch.no_grad()
def test_step(features, labels):

    logits_model.eval()

    logits = logits_model(features)

    loss = criterion(logits, labels)

    metrics_dict["test_loss"].update(loss.item())


""" AUXILIARY FUNCTIONS """

def calculate_accuracy_for_each_class(predictions, labels_param):

    probs = torch.sigmoid(predictions)

    for i, class_name in enumerate(CLASSES):

        class_labels = labels_param[:, i]

        y_pred = probs[:, i]
        y_pred = (y_pred > 0.5).float()

        accuracy = (y_pred == class_labels).float().mean()

        metrics_dict[f"{class_name.lower()}_train_accuracy"].update(
            accuracy.item()
        )


def write_results_description_in_txt(txt_file):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")




""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file)

    csv_file = open(csv_path, "w+")
    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    print(f"starting epoch {epoch}")
    #reset metrics
    for metrics in metrics_dict.values():
        metrics.reset_state()


    for features_param, labels in train_loader:

      features_param = features_param.to(device)
      labels = labels.to(device)

      train_step(features_param, labels)


    for features_param, labels in test_loader:

        features_param = features_param.to(device)
        labels = labels.to(device)

        test_step(features_param, labels)

    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()

torch.save(logits_model.state_dict(), f"model_baseline_epoch_{epoch}.pth")

if csv_path is not None:
    csv_file.close()
    txt_file.close()

starting epoch 0
current loss: 0.6771466732025146
current loss: 0.6567100286483765
current loss: 0.5981974601745605
current loss: 0.5395768880844116
current loss: 0.5126150846481323
current loss: 0.66026771068573
current loss: 0.616314709186554
current loss: 0.5390615463256836
current loss: 0.6431362628936768
current loss: 0.4919160008430481
current loss: 0.5806919932365417
current loss: 0.5475058555603027
current loss: 0.6184510588645935
current loss: 0.5247862339019775
current loss: 0.5619179010391235
current loss: 0.5090570449829102
current loss: 0.47366923093795776
current loss: 0.4436870813369751
current loss: 0.6050456166267395
current loss: 0.5044364333152771
current loss: 0.5341821312904358
current loss: 0.49942341446876526
current loss: 0.54839026927948
current loss: 0.4972171187400818
current loss: 0.5163198709487915


começo do codigo

In [ ]:

data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(DATA_ROOT, "all.csv")
images = Path(DATA_ROOT, "c1")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]
CLASSES = [
    "Cafe",
    "Hotel",
    "Store",
    "MiscCommercial",
    "Awning",
    "Billboard",
    "VendingMachine",
    "Statue",
    "Table",
    "WallSign",
]

class BuildingsDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = self.img_dir / row["id"]
        image = Image.open(img_path).convert("RGB")



        #target = torch.tensor(row[CLASSES].values, dtype=torch.float32)
        target = torch.tensor(row[CLASSES].to_numpy(dtype=float), dtype=torch.float32)

        #print(row[CLASSES].to_numpy().dtype)


        if self.transform:
            image = self.transform(image)

        return image, target


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


TRAIN_TEST_LIMIT = int(0.8 * len(df))

train_df = df[:TRAIN_TEST_LIMIT]
test_df = df[TRAIN_TEST_LIMIT:]

df[CLASSES] = df[CLASSES].apply(pd.to_numeric)


train_dataset = BuildingsDataset(train_df, images, transform=transform)
test_dataset = BuildingsDataset(test_df, images, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)



CNN model

In [ ]:
#print(train_df["Cafe"])

"""CNN MODEL"""
class CNN(nn.Module):
   def __init__(self, in_channels, num_classes, name="cnn_model"):

       """
       Building blocks of convolutional neural network.

       Parameters:
           * in_channels: Number of channels in the input image (for grayscale images, 1)
           * num_classes: Number of classes to predict. In our problem, 10 (i.e digits from  0 to 9).
       """
       super(CNN, self).__init__()

       #layers
       self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3)
       self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3)
       self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)

       self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
       self.flatten = nn.Flatten()
       self.dropout = nn.Dropout(0.5)

       self.dense1 = nn.Linear(43264, 128)
       self.dense2 = nn.Linear(128, 64)
       self.concept_layer = nn.Linear(64, 32)
       self.output_layer = nn.Linear(32, num_classes)


   def forward(self, inputs):
       """
       Define the forward pass of the neural network.

       Parameters:
           x: Input tensor.

       Returns:
           torch.Tensor
               The output tensor after passing through the network.
       """
       if isinstance(inputs, (list, tuple)):
          inputs = inputs[0]


       x = F.relu(self.conv1(inputs))
       x = self.maxpool(x)
       x = F.relu(self.conv2(x))
       x = self.maxpool(x)
       x = F.relu(self.conv3(x))
       x = self.maxpool(x)

       x = self.flatten(x)
       x = self.dropout(x)

       x = F.relu(self.dense1(x))
       x = self.dropout(x)
       x = F.relu(self.dense2(x))

       y = F.relu(self.concept_layer(x))
       x = self.output_layer(y)

       result = torch.cat([x, y], dim=1)
       return result


class PredicateModel(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.activation = nn.Sigmoid()

    def forward(self, x, class_idx):
        logits = self.model(x)              # [batch, num_classes]
        probs = self.activation(logits)     # [0,1]
        print("shape in prpbs")
        print(probs.shape)
        #print("shape in predicate model")
        #print(probs[:, class_idx].squeeze().shape)

        if class_idx.dim() == 0:
            class_idx = class_idx.repeat(x.shape[0])

        class_idx = class_idx.long().unsqueeze(1)

        selected = torch.gather(probs, 1, class_idx)

        return selected.squeeze(1)      # [batch]






LTN Model

In [ ]:


# Predicates
logits_model = CNN(in_channels=3, num_classes=4)

# automatically use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# move model to device
logits_model = logits_model.to(device)

p = ltn.Predicate(PredicateModel(logits_model))

#print("before constants")

# Constants
class_cafe = ltn.Constant(torch.tensor(CLASSES.index("Cafe")), trainable=False)
class_hotel = ltn.Constant(torch.tensor(CLASSES.index("Hotel")), trainable=False)
class_store = ltn.Constant(torch.tensor(CLASSES.index("Store")), trainable=False)
class_misc_commercial = ltn.Constant(torch.tensor(CLASSES.index("MiscCommercial")), trainable=False)
class_awning = ltn.Constant(torch.tensor(CLASSES.index("Awning")), trainable=False)
class_billboard = ltn.Constant(torch.tensor(CLASSES.index("Billboard")), trainable=False)
class_vending_machine = ltn.Constant(torch.tensor(CLASSES.index("VendingMachine")), trainable=False)
class_statue = ltn.Constant(torch.tensor(CLASSES.index("Statue")), trainable=False)
class_table = ltn.Constant(torch.tensor(CLASSES.index("Table")), trainable=False)
class_wall_sign = ltn.Constant(torch.tensor(CLASSES.index("WallSign")), trainable=False)


#Connectives
Not = ltn.Connective(ltn.fuzzy_ops.NotStandard())
And = ltn.Connective(ltn.fuzzy_ops.AndProd())
Or = ltn.Connective(ltn.fuzzy_ops.OrProbSum())
Implies = ltn.Connective(ltn.fuzzy_ops.ImpliesReichenbach())
Forall = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")
Exists = ltn.Quantifier(ltn.fuzzy_ops.AggregPMean(p=2), quantifier="e")
Equiv = ltn.Connective(
    ltn.fuzzy_ops.Equiv(
        ltn.fuzzy_ops.AndProd(),
        ltn.fuzzy_ops.ImpliesReichenbach()
    )
)

formula_aggregator = ltn.fuzzy_ops.SatAgg(
    ltn.fuzzy_ops.AggregPMeanError(p=2)
)


#axioms
def axioms(features_param, *args):

    x = ltn.Variable("x", features_param)
    x_hotel = ltn.Variable("x_hotel", features_param[args[CLASSES.index("Hotel")] == 1, :])
    x_not_hotel = ltn.Variable(
        "x_not_hotel", features_param[args[CLASSES.index("Hotel")] == 0, :]
    )
    x_cafe = ltn.Variable("x_cafe", features_param[args[CLASSES.index("Cafe")] == 1, :])
    x_not_cafe = ltn.Variable(
        "x_not_cafe", features_param[args[CLASSES.index("Cafe")] == 0, :]
    )
    x_store = ltn.Variable("x_store", features_param[args[CLASSES.index("Store")] == 1, :])
    x_not_store = ltn.Variable(
        "x_not_store", features_param[args[CLASSES.index("Store")] == 0, :]
    )
    x_misc_commercial = ltn.Variable(
        "x_misc_commercial", features_param[args[CLASSES.index("MiscCommercial")] == 1, :]
    )
    x_not_misc_commercial = ltn.Variable(
        "x_not_misc_commercial",
        features_param[args[CLASSES.index("MiscCommercial")] == 0, :],
    )

    axioms = [
        # Grounded Predicates


        Forall(x_hotel, p(x_hotel, class_hotel)),
        Forall(x_not_hotel, Not(p(x_not_hotel, class_hotel))),
        Forall(x_cafe, p(x_cafe, class_cafe)),
        Forall(x_not_cafe, Not(p(x_not_cafe, class_cafe))),
        Forall(x_store, p(x_store, class_store)),
        Forall(x_not_store, Not(p(x_not_store, class_store))),
        Forall(x_misc_commercial, p(x_misc_commercial, class_misc_commercial)),
        Forall(
            x_not_misc_commercial,
            Not(p(x_not_misc_commercial, class_misc_commercial)),
        ),
        # Axioms
        # Cafe = Statue or Vending Machine
        Forall(
            x,
            Equiv(
                p(x, class_cafe),
                Or(p(x, class_statue), p(x, class_vending_machine)),
            ),
        ),
        # Hotel = Wall Sign
        Forall(
            x,
            Equiv(p(x, class_hotel), p(x, class_wall_sign)),
        ),
        # MiscCommercial = Awning and Table
        Forall(
            x,
            Equiv(
                p(x, class_misc_commercial),
                And(p(x, class_awning), p(x, class_table)),
            ),
        ),
        # Store = Billboard
        Forall(
            x,
            Equiv(p(x, class_store), p(x, class_billboard)),
        ),
    ]
    sat_level = formula_aggregator(*axioms)
    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list

print("before for loop")

for batch_elements in train_loader:
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(features, *labels_list))
    break



before for loop
shape in prpbs
torch.Size([7, 36])
shape in prpbs
torch.Size([25, 36])
shape in prpbs
torch.Size([7, 36])
shape in prpbs
torch.Size([25, 36])
shape in prpbs
torch.Size([3, 36])
shape in prpbs
torch.Size([29, 36])
shape in prpbs
torch.Size([4, 36])
shape in prpbs
torch.Size([28, 36])
shape in prpbs
torch.Size([32, 36])
shape in prpbs
torch.Size([32, 36])
shape in prpbs
torch.Size([32, 36])
shape in prpbs
torch.Size([32, 36])
shape in prpbs
torch.Size([32, 36])
shape in prpbs
torch.Size([32, 36])
shape in prpbs
torch.Size([32, 36])
shape in prpbs
torch.Size([32, 36])
shape in prpbs
torch.Size([32, 36])
shape in prpbs
torch.Size([32, 36])
Initial sat level 0.51488


In [ ]:
""" TRAINING FUNCTIONS """

class MeanMetric:
    def __init__(self):
        self.reset_state()

    def update(self, value, n=1):
        self.total += value * n
        self.count += n

    def compute(self):
        return self.total / self.count if self.count != 0 else 0.0

    def reset_state(self):
        self.total = 0.0
        self.count = 0

    def result(self):
        return self.compute()

metrics_dict = {
    "train_sat_kb": MeanMetric(),
    "test_sat_kb": MeanMetric(),
}

class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": MeanMetric()
    for class_name in CLASSES
}

metrics_dict.update(class_train_accuracy_metric)

optimizer = torch.optim.Adam(logits_model.parameters(), lr=LEARNING_RATE)


def train_step(features_param, *labels_list):
    logits_model.train()

    optimizer.zero_grad()

    # Forward through axioms (LTN)
    sat = axioms(features_param, *labels_list)

    loss = 1.0 - sat

    # Backprop
    loss.backward()
    optimizer.step()

    print("Sat Function Result:", sat.item())

    # Update metric
    metrics_dict["train_sat_kb"].update(sat.item())

    # Predictions
    predictions = logits_model(features_param)

    calculate_accuracy_for_each_class(predictions, labels_list)

@torch.no_grad()
def test_step(features_param, *labels_list):
    logits_model.eval()

    sat_kb = axioms(features_param, *labels_list)

    metrics_dict["test_sat_kb"].update(sat_kb.item())


""" AUXILIARY FUNCTIONS """

def calculate_accuracy_for_each_class(predictions, labels_param):
    probs = torch.sigmoid(predictions)

    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i].float()

        y_pred = probs[:, i]
        y_pred = (y_pred > 0.5).float()

        accuracy = (y_pred == class_labels).float().mean()

        metrics_dict[f"{class_name.lower()}_train_accuracy"].update(
            accuracy.item()
        )


def write_results_description_in_txt(txt_file):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")

In [ ]:
""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file)

    csv_file = open(csv_path, "w+")
    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    print(f"starting epoch {epoch}")
    #reset metrics
    for metrics in metrics_dict.values():
        metrics.reset_state()

    for batch_elements in train_loader:
        features_param, labels_list = __get_elements_from_batch(batch_elements)

        # Move to device if needed
        features_param = features_param.to(device)
        labels_list = [lbl.to(device) for lbl in labels_list]

        train_step(features_param, *labels_list)


    for batch_elements_test in test_loader:
        features_param, labels_list = __get_elements_from_batch(batch_elements_test)

        # Move to device if needed
        features_param = features_param.to(device)
        labels_list = [lbl.to(device) for lbl in labels_list]

        test_step(features_param, *labels_list)

    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()


if csv_path is not None:
    csv_file.close()
    txt_file.close()


starting epoch 0
features param shape
torch.Size([32, 3, 224, 224])
Sat Function Result: 0.5196667909622192
features param shape
torch.Size([32, 3, 224, 224])
Sat Function Result: 0.5195386409759521
features param shape
torch.Size([32, 3, 224, 224])
Sat Function Result: 0.5190877914428711
features param shape
torch.Size([32, 3, 224, 224])
Sat Function Result: 0.5190682411193848
features param shape
torch.Size([32, 3, 224, 224])
Sat Function Result: 0.5197278261184692
features param shape
torch.Size([32, 3, 224, 224])
Sat Function Result: 0.519722044467926
features param shape
torch.Size([32, 3, 224, 224])
Sat Function Result: 0.5202361345291138
features param shape
torch.Size([32, 3, 224, 224])
Sat Function Result: 0.5205261707305908
features param shape
torch.Size([32, 3, 224, 224])
Sat Function Result: 0.5206750631332397
features param shape
torch.Size([32, 3, 224, 224])
Sat Function Result: 0.5203545093536377
features param shape
torch.Size([32, 3, 224, 224])
Sat Function Result: 0.

KeyboardInterrupt: 